***Step 1: Spatial Attribute***

In [2]:
import os
import matplotlib as mpl
import matplotlib.pyplot as plt

# --------- JOURNAL FIGURE SETTINGS ----------
FIG_DIR = "figures_journal"   # all images will be saved here
os.makedirs(FIG_DIR, exist_ok=True)

mpl.rcParams['savefig.dpi'] = 300
mpl.rcParams['figure.dpi'] = 300
mpl.rcParams['font.family'] = 'Times New Roman'  # common journal preference
mpl.rcParams['font.size'] = 10
mpl.rcParams['axes.titlesize'] = 11
mpl.rcParams['axes.labelsize'] = 10
mpl.rcParams['legend.fontsize'] = 9
mpl.rcParams['xtick.labelsize'] = 9
mpl.rcParams['ytick.labelsize'] = 9
mpl.rcParams['lines.linewidth'] = 1.0

def save_figure(fig, name, dpi=300):
    """
    Saves both:
    - PNG at dpi (good for journals requiring raster)
    - PDF vector (best for floor plans/graphs)
    """
    png_path = os.path.join(FIG_DIR, f"{name}.png")
    pdf_path = os.path.join(FIG_DIR, f"{name}.pdf")
    fig.tight_layout()
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print(f"✅ Saved: {png_path}")
    print(f"✅ Saved: {pdf_path}")
# -------------------------------------------


In [3]:
import numpy as np
from shapely.geometry import Polygon

def get_plot_input():
    plot_type = input("Enter plot type (rectangular/irregular): ").strip().lower()

    if plot_type == "rectangular":
        length = float(input("Enter plot length (in meters): "))
        width = float(input("Enter plot width (in meters): "))
        
        # Define boundary coordinates (clockwise)
        coordinates = [(0, 0), (length, 0), (length, width), (0, width)]
    
    elif plot_type == "irregular":
        print("Enter corner coordinates of the plot in format x,y (type 'done' to finish):")
        coordinates = []
        while True:
            user_input = input("Enter coordinate: ")
            if user_input.lower() == 'done':
                break
            try:
                x, y = map(float, user_input.strip().split(','))
                coordinates.append((x, y))
            except ValueError:
                print("Invalid input. Please enter in format x,y")
        
        if len(coordinates) < 3:
            raise ValueError("Polygon must have at least 3 coordinates.")

    else:
        raise ValueError("Invalid plot type. Must be 'rectangular' or 'irregular'.")

    # Create polygon and bounding info
    plot_polygon = Polygon(coordinates)
    bounds = plot_polygon.bounds  # (minx, miny, maxx, maxy)

    # Store spatial attributes
    spatial_attributes = {
        'plot_type': plot_type,
        'coordinates': coordinates,
        'polygon': plot_polygon,
        'bounds': bounds,
        'grid_origin': (0, 0),
        'direction_vectors': {
            'right': (1, 0),
            'left': (-1, 0),
            'up': (0, 1),
            'down': (0, -1)
        }
    }

    return spatial_attributes

In [4]:
spatial_data = get_plot_input()

In [ ]:
def visualize_plot(spatial_data, step_tag="Step1_PlotBoundary", show_grid=False):
    polygon = spatial_data['polygon']
    x, y = polygon.exterior.xy

    fig, ax = plt.subplots(figsize=(6, 6))  # journal-friendly size

    ax.plot(x, y, color='black', linewidth=2, label='Plot Boundary')
    ax.fill(x, y, alpha=0.1)

    # vertex indices
    for i, (xi, yi) in enumerate(zip(x, y)):
        ax.text(xi, yi, f"{i}", fontsize=10, ha='center', va='center', color='blue')

    ax.set_title(f"Plot Shape: {spatial_data['plot_type'].capitalize()}")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_aspect("equal", adjustable="box")

    if show_grid:
        ax.grid(True, linewidth=0.3, alpha=0.4)  # lighter grid if you keep it

    ax.legend(frameon=False)

    # Save journal-quality outputs
    save_figure(fig, step_tag)

    plt.show()

visualize_plot(spatial_data)

visualize_plot(spatial_data, show_grid=True)


***Step 1.2: Component Attributes***

In [6]:
import numpy as np

# 1. Define axis mapping rules
dimension_axis_map = {
    "wall": {
        "length": ["x", "y"],
        "width": ["z"],
        "thickness": ["x", "y"]
    },
    "floor": {
        "length": ["x", "y"],
        "width": ["x", "y"],
        "thickness": ["z"]
    },
    "slab": {
        "length": ["x", "y"],
        "width": ["x", "y"],
        "thickness": ["z"]
    },
    "door": {
        "length": ["x", "y"],
        "width": ["z"],
        "thickness": ["x", "y"]
    },
    "window": {
        "length": ["x", "y"],
        "width": ["z"],
        "thickness": ["x", "y"]
    }
}

# 2. Define component class
class BuildingComponent:
    def __init__(self, obj_type, length, width, thickness, shape="rectangle"):
        self.obj_type = obj_type.lower()
        self.length = length
        self.width = width
        self.thickness = thickness
        self.shape = shape.lower()

        self.dimension_axes = dimension_axis_map.get(self.obj_type, {})

        self.area = self.compute_area()
        self.volume = self.compute_volume()
        self.centroid = self.compute_centroid()

    def compute_area(self):
        if self.shape == "rectangle":
            return self.length * self.width
        elif self.shape == "circle":
            return np.pi * (self.length / 2) ** 2
        else:
            return None

    def compute_volume(self):
        return self.area * self.thickness if self.area is not None else None

    def compute_centroid(self):
        return (self.length / 2, self.width / 2, self.thickness / 2)

    def move_to(self, x, y, z=0):
        self.centroid = (x + self.length / 2, y + self.width / 2, z + self.thickness / 2)

    def validate_alignment(self, direction):
        valid = {
            'x': "x" in self.dimension_axes.get("length", []) or 
                 "x" in self.dimension_axes.get("width", []) or 
                 "x" in self.dimension_axes.get("thickness", []),
            'y': "y" in self.dimension_axes.get("length", []) or 
                 "y" in self.dimension_axes.get("width", []) or 
                 "y" in self.dimension_axes.get("thickness", []),
            'z': "z" in self.dimension_axes.get("length", []) or 
                 "z" in self.dimension_axes.get("width", []) or 
                 "z" in self.dimension_axes.get("thickness", [])
        }
        return valid.get(direction.lower(), False)

    def describe(self):
        return {
            "type": self.obj_type,
            "dimensions": {
                "length": self.length,
                "width": self.width,
                "thickness": self.thickness
            },
            "shape": self.shape,
            "allowed_axes": self.dimension_axes,
            "area": self.area,
            "volume": self.volume,
            "centroid": self.centroid
        }

In [7]:
def create_component_from_input():
    obj_type = input("Enter object type (wall, floor, slab, door, window): ").lower()
    length = float(input("Enter length (m): "))
    width = float(input("Enter width (m): "))
    thickness = float(input("Enter thickness (m): "))
    shape = input("Enter shape (rectangle, circle, etc.): ").strip().lower()

    return BuildingComponent(obj_type, length, width, thickness, shape)

In [11]:
print("🔸 Create Wall")
wall = create_component_from_input()

print("\n🔸 Create Floor")
floor = create_component_from_input()

print("\n🔸 Create Slab")
slab = create_component_from_input()

print("\n🔸 Create Door")
door = create_component_from_input()

print("\n🔸 Create Window")
window = create_component_from_input()

# Store all in dictionary
components = {
    'wall': wall,
    'floor': floor,
    'slab': slab,
    'door': door,
    'window': window
}

🔸 Create Wall

🔸 Create Floor

🔸 Create Slab

🔸 Create Door

🔸 Create Window


In [12]:
# Example placement positions
placements = {
    'wall': (1, 1, 0),
    'floor': (4, 1, 0),
    'slab': (7, 1, 0),
    'door': (1, 4, 0),
    'window': (4, 4, 0)
}

# Move components
for name, comp in components.items():
    comp.move_to(*placements[name])

In [ ]:
import matplotlib.patches as patches

def visualize_components(spatial_data, components, step_tag="Step1_2_Components", show_grid=False):
    fig, ax = plt.subplots(figsize=(7, 5.5))  # journal-friendly size

    # Plot the boundary
    polygon = spatial_data['polygon']
    x, y = polygon.exterior.xy
    ax.plot(x, y, color='black', linewidth=2, label="Plot Boundary")
    ax.fill(x, y, alpha=0.05, color='gray')

    for name, comp in components.items():
        cx, cy, cz = comp.centroid

        # Determine 2D visible dimensions
        if comp.obj_type in ['wall', 'door', 'window']:
            w = comp.length
            h = comp.thickness
        else:  # floor, slab
            w = comp.length
            h = comp.width

        lower_left_x = cx - w / 2
        lower_left_y = cy - h / 2

        rect = patches.Rectangle(
            (lower_left_x, lower_left_y), w, h,
            linewidth=1.2,
            edgecolor='blue',
            facecolor='skyblue',
            alpha=0.6
        )
        ax.add_patch(rect)

        ax.text(cx, cy, name.upper(), ha='center', va='center', fontsize=9, color='black')

    ax.set_aspect('equal', adjustable='box')
    ax.set_title("Component Plan View (2D)")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")

    if show_grid:
        ax.grid(True, linewidth=0.3, alpha=0.4)

    # Save as journal-quality PNG + vector PDF
    save_figure(fig, step_tag)

    plt.show()

visualize_components(spatial_data, components)

visualize_components(spatial_data, components, show_grid=True)


***Step 2: Functional Area Attribute***

Step 2.1: Define Room Type Constraint

In [14]:
def collect_room_constraints_from_user():
    print("📋 Enter room types and their constraints (area, length, width, enclosure).")
    print("Use the format: Room Name, Area, Length, Width, Enclosure Type")
    print("Example: Master Bedroom (no cabinet), 12, 3.5, 3, internal")
    print("Type 'done' when finished.\n")

    room_constraints = {}
    while True:
        line = input("Enter room constraint or 'done': ").strip()
        if line.lower() == 'done':
            break
        try:
            name, area, length, width, enclosure = [x.strip() for x in line.split(',')]
            room_constraints[name] = {
                'area': float(area),
                'length': float(length),
                'width': float(width),
                'enclosure': enclosure.lower()
            }
        except ValueError:
            print("⚠️ Invalid input format. Please follow the example.")
    
    print("\n✅ Room constraints successfully recorded.\n")
    return room_constraints

# Run this at the start of Step 2 to generate user-defined room constraints:
room_constraints = collect_room_constraints_from_user()


📋 Enter room types and their constraints (area, length, width, enclosure).
Use the format: Room Name, Area, Length, Width, Enclosure Type
Example: Master Bedroom (no cabinet), 12, 3.5, 3, internal
Type 'done' when finished.


✅ Room constraints successfully recorded.



Step 2.2: Room Class with Constraint Validation

In [15]:
class Room:
    def __init__(self, name, constraints, components_needed=None):
        self.name = name
        self.area_min = constraints["area"]
        self.length_min = constraints["length"]
        self.width_min = constraints["width"]
        self.enclosure = constraints["enclosure"]
        self.components = components_needed or ["walls", "door", "floor"]

        # Placeholder for actual placement
        self.assigned = False
        self.position = None
        self.orientation = None

    def describe(self):
        return {
            "Room Type": self.name,
            "Min Area": self.area_min,
            "Min Length": self.length_min,
            "Min Width": self.width_min,
            "Enclosure": self.enclosure,
            "Components": self.components,
            "Assigned": self.assigned
        }

Step 2.3: Input for Room Quantity

In [16]:
def get_room_requirements(room_constraints):
    room_plan = {}
    print("Enter how many of each room type you want (0 to skip):")
    for name in room_constraints:
        count = int(input(f"{name}: "))
        if count > 0:
            room_plan[name] = count
    return room_plan

Step 2.4: Adjacency and Connectivity Matrices

In [17]:
import pandas as pd

def get_adjacency_matrix(room_types):
    print("\nDefine Room Adjacency (1 = adjacent, 0 = not):")
    matrix = pd.DataFrame(index=room_types, columns=room_types)
    for i in room_types:
        for j in room_types:
            if i == j:
                matrix.loc[i, j] = 0
            else:
                matrix.loc[i, j] = int(input(f"Is '{i}' adjacent to '{j}'? (0/1): "))
    return matrix

def get_connectivity_matrix(room_types):
    print("\nDefine Room Connectivity (0 = not, 1/2/3 = increasing distance):")
    matrix = pd.DataFrame(index=room_types, columns=room_types)
    for i in room_types:
        for j in room_types:
            if i == j:
                matrix.loc[i, j] = 0
            else:
                matrix.loc[i, j] = int(input(f"Connectivity level from '{i}' to '{j}' (0-3): "))
    return matrix

Trigger Input For Rooms

In [18]:
# Step 1: Ask how many of each room type
room_plan = get_room_requirements(room_constraints)

# Step 2: Generate Room objects
rooms = []
for name, count in room_plan.items():
    for _ in range(count):
        room = Room(name, room_constraints[name])
        rooms.append(room)

# Step 3: Get list of room names (simplified)
room_names = [f"{r.name}" for r in rooms]
unique_room_names = list(set(room_names))

# Step 4: Get adjacency and connectivity
adjacency_matrix = get_adjacency_matrix(unique_room_names)
connectivity_matrix = get_connectivity_matrix(unique_room_names)

Enter how many of each room type you want (0 to skip):

Define Room Adjacency (1 = adjacent, 0 = not):

Define Room Connectivity (0 = not, 1/2/3 = increasing distance):


Visualzation

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def visualize_room_graph(adjacency_matrix,
                         connectivity_matrix=None,
                         room_plan=None,
                         show_weights=True,
                         step_tag="Step2_RoomRelationshipGraph"):

    G = nx.Graph()

    # Add nodes (rooms)
    for room in adjacency_matrix.index:
        label = room
        if room_plan and room in room_plan:
            try:
                if isinstance(room_plan[room], dict):
                    label += f" ({room_plan[room]['count']})"
                else:
                    label += f" ({room_plan[room]})"
            except Exception:
                pass
        G.add_node(room, label=label)

    # Add edges from adjacency
    for i in adjacency_matrix.index:
        for j in adjacency_matrix.columns:
            if i != j and int(adjacency_matrix.loc[i, j]) == 1:
                weight = None
                if connectivity_matrix is not None:
                    weight = int(connectivity_matrix.loc[i, j])
                G.add_edge(i, j, weight=weight)

    # Layout and labels
    pos = nx.spring_layout(G, seed=42)  # reproducible layout
    labels = nx.get_node_attributes(G, 'label')
    edge_weights = nx.get_edge_attributes(G, 'weight')

    # ---- Journal-friendly figure ----
    fig, ax = plt.subplots(figsize=(7.5, 5.5))  # good for single-column or 1.5-column papers

    nx.draw(
        G, pos,
        ax=ax,
        with_labels=True,
        labels=labels,
        node_size=1800,
        node_color='lightblue',
        font_size=9,
        font_weight='bold',
        edge_color='gray',
        width=1.0
    )

    # Draw edge weights if available
    if connectivity_matrix is not None and show_weights:
        nx.draw_networkx_edge_labels(
            G, pos,
            ax=ax,
            edge_labels=edge_weights,
            font_size=8
        )

    ax.set_title("Room Relationship Graph (Adjacency & Connectivity)")
    ax.axis('off')

    # Save as PNG (300 dpi) + PDF (vector)
    save_figure(fig, step_tag)

    plt.show()

visualize_room_graph(adjacency_matrix, connectivity_matrix, room_plan)

pos = nx.spring_layout(G, seed=42, k=1.2)

fig, ax = plt.subplots(figsize=(8.5, 6))


***Step 3: Creating Building Area and Grids***

In [ ]:
# ============================================================
# STEP 3: Creating Building Area and Grids (REPRODUCIBLE)
# - Option A: Fixed SEED + Fixed candidate index
# - No changes needed in Step 1, Step 2, Step 4–Step 10
# ============================================================

import matplotlib.pyplot as plt
from shapely.geometry import Polygon, box
import random
import numpy as np

# -------------------- BENCHMARK CONTROLS (ADD THESE) --------------------
BENCHMARK_MODE = True          # True = always pick the same candidate automatically
BENCHMARK_SEED = 2026          # keep SAME number in GA/ACO/PSO/SA files
FIXED_OPTION_INDEX = 1         # always select Option 1 (1..num_candidates)

# Set seeds ONCE before any random candidate generation happens
random.seed(BENCHMARK_SEED)
np.random.seed(BENCHMARK_SEED)
# ----------------------------------------------------------------------


def input_far_and_offset():
    far = float(input("Enter Floor Area Ratio (FAR): "))
    offset_left = float(input("Offset from left edge (m): "))
    offset_bottom = float(input("Offset from bottom edge (m): "))
    offset_right = float(input("Offset from right edge (m): "))
    offset_top = float(input("Offset from top edge (m): "))
    return far, offset_left, offset_bottom, offset_right, offset_top


def shrink_polygon_by_offsets(polygon, left, bottom, right, top):
    minx, miny, maxx, maxy = polygon.bounds
    return box(minx + left, miny + bottom, maxx - right, maxy - top)


def generate_candidate_building_areas_with_offsets(polygon, far, offset_values, num_candidates=5):
    left, bottom, right, top = offset_values
    offset_polygon = shrink_polygon_by_offsets(polygon, left, bottom, right, top)

    plot_area = polygon.area
    offset_area = offset_polygon.area
    max_area = far * plot_area

    fallback_triggered = False
    if offset_area < max_area:
        print("⚠️ Offset region too small for FAR requirement. Using full plot as fallback.")
        offset_polygon = polygon
        fallback_triggered = True

    candidates = []
    tries = 0

    # NOTE: Because random.seed(...) is set above, the sequence here is reproducible
    while len(candidates) < num_candidates and tries < 200:
        tries += 1

        w_max = offset_polygon.bounds[2] - offset_polygon.bounds[0]
        h_max = offset_polygon.bounds[3] - offset_polygon.bounds[1]

        w = random.uniform(1.5, w_max)
        h = max_area / w

        if h > h_max:
            continue

        x = random.uniform(offset_polygon.bounds[0], offset_polygon.bounds[2] - w)
        y = random.uniform(offset_polygon.bounds[1], offset_polygon.bounds[3] - h)

        candidate = box(x, y, x + w, y + h)

        if offset_polygon.contains(candidate) and candidate.area <= max_area:
            candidates.append(candidate)

    if len(candidates) < num_candidates:
        print(f"⚠️ Only generated {len(candidates)} candidate(s) after {tries} tries. "
              f"Consider adjusting FAR/offsets or increasing tries.")

    return offset_polygon, candidates


def visualize_building_candidates(plot_polygon, offset_polygon, candidates,
                                  step_tag="Step3_CandidateBuildingAreas",
                                  show_grid=False):
    fig, ax = plt.subplots(figsize=(7.5, 7.5))  # journal-friendly

    x, y = plot_polygon.exterior.xy
    ax.plot(x, y, color='gray', linestyle='--', linewidth=1.0, label="Plot Area")

    ox, oy = offset_polygon.exterior.xy
    ax.plot(ox, oy, color='black', linestyle=':', linewidth=1.0, label="Offset Region")

    colors = ['lightblue', 'lightsalmon', 'lightgreen', 'lightpink', 'plum']
    for idx, c in enumerate(candidates):
        bx, by = c.exterior.xy
        ax.fill(bx, by, color=colors[idx % len(colors)], alpha=0.35, label=f"Option {idx+1}")
        cx, cy = c.centroid.x, c.centroid.y
        ax.text(cx, cy, str(idx+1), fontsize=9, ha='center', va='center')

    ax.set_aspect('equal', adjustable='box')
    ax.set_title("Candidate Building Areas with Edge Offsets")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.legend(frameon=False, loc="best")

    if show_grid:
        ax.grid(True, linewidth=0.3, alpha=0.4)

    # Save 300 dpi PNG + vector PDF
    save_figure(fig, step_tag)

    plt.show()



def choose_building_area(candidates):
    while True:
        try:
            idx = int(input(f"Select a building option (1-{len(candidates)}): "))
            if 1 <= idx <= len(candidates):
                return candidates[idx - 1]
            else:
                print("Invalid option.")
        except:
            print("Enter a valid integer.")


def input_grid_size():
    grid_x = float(input("Enter grid cell width (m): "))
    grid_y = float(input("Enter grid cell height (m): "))
    return grid_x, grid_y


def divide_building_area_into_grid(building_area, grid_x, grid_y):
    minx, miny, maxx, maxy = building_area.bounds
    x_vals = np.arange(minx, maxx, grid_x)
    y_vals = np.arange(miny, maxy, grid_y)

    grid_cells = []
    for x in x_vals:
        for y in y_vals:
            cell = box(x, y, x + grid_x, y + grid_y)
            if building_area.contains(cell):
                grid_cells.append(cell)
    return grid_cells


def visualize_final_grid(plot_polygon, building_area, grid_cells,
                         step_tag="Step3_FinalGrid",
                         show_grid=False):
    fig, ax = plt.subplots(figsize=(7.5, 7.5))

    x, y = plot_polygon.exterior.xy
    ax.plot(x, y, color='gray', linestyle='--', linewidth=1.0, label="Plot")

    bx, by = building_area.exterior.xy
    ax.plot(bx, by, color='black', linewidth=1.2, label="Selected Building Area")

    for c in grid_cells:
        gx, gy = c.exterior.xy
        ax.plot(gx, gy, color='blue', linewidth=0.4)  # thinner looks cleaner in print

    ax.set_aspect('equal', adjustable='box')
    ax.set_title("Final Grid within Selected Building Area")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.legend(frameon=False, loc="best")

    if show_grid:
        ax.grid(True, linewidth=0.3, alpha=0.4)

    # Save 300 dpi PNG + vector PDF
    save_figure(fig, step_tag)

    plt.show()



# --- STEP 3: Input and Generate Building Area with Offset Constraint ---
far, offset_left, offset_bottom, offset_right, offset_top = input_far_and_offset()

offset_polygon, candidates = generate_candidate_building_areas_with_offsets(
    spatial_data['polygon'], far, (offset_left, offset_bottom, offset_right, offset_top),
    num_candidates=5
)

visualize_building_candidates(spatial_data['polygon'], offset_polygon, candidates)

# -------------------- FIXED SELECTION FOR BENCHMARKING --------------------
if BENCHMARK_MODE:
    if len(candidates) == 0:
        raise ValueError("No candidates were generated. Adjust FAR/offsets or increase tries.")
    if not (1 <= FIXED_OPTION_INDEX <= len(candidates)):
        raise ValueError(f"FIXED_OPTION_INDEX={FIXED_OPTION_INDEX} is out of range. "
                         f"Generated {len(candidates)} candidates.")
    selected_building = candidates[FIXED_OPTION_INDEX - 1]
    print(f"✅ BENCHMARK MODE: Selected fixed Option {FIXED_OPTION_INDEX} (seed={BENCHMARK_SEED})")
else:
    selected_building = choose_building_area(candidates)
# -------------------------------------------------------------------------


grid_x, grid_y = input_grid_size()
grid_cells = divide_building_area_into_grid(selected_building, grid_x, grid_y)
visualize_final_grid(spatial_data['polygon'], selected_building, grid_cells)


***Step 4: Hybrid Graph Data Representation***

Step 4.1: Graph Data Representation of Grids

Create Intersection Graph

In [21]:
import networkx as nx
import matplotlib.pyplot as plt
from shapely.geometry import Point

def get_intersections_from_cells(grid_cells):
    intersections = set()
    for cell in grid_cells:
        minx, miny, maxx, maxy = cell.bounds
        intersections.update([
            (round(minx, 6), round(miny, 6)),
            (round(minx, 6), round(maxy, 6)),
            (round(maxx, 6), round(miny, 6)),
            (round(maxx, 6), round(maxy, 6))
        ])
    # Sort bottom to top, left to right within each row
    sorted_pts = sorted(list(intersections), key=lambda p: (round(p[1], 6), round(p[0], 6)))
    return sorted_pts

def generate_intersection_graph_from_cells(grid_cells, tol=1e-4):
    intersections = get_intersections_from_cells(grid_cells)
    G = nx.Graph()
    node_id_map = {}

    # Add nodes with row-wise numbering
    for idx, pt in enumerate(intersections):
        G.add_node(idx, pos=pt)
        node_id_map[pt] = idx

    # Determine spacing from first cell
    minx, miny, maxx, maxy = grid_cells[0].bounds
    dx = round(maxx - minx, 6)
    dy = round(maxy - miny, 6)

    # Connect neighbors in 4 directions
    for pt in intersections:
        x, y = pt
        neighbors = [
            (round(x + dx, 6), y),
            (x, round(y + dy, 6)),
            (round(x - dx, 6), y),
            (x, round(y - dy, 6))
        ]
        for nbr in neighbors:
            for other_pt in intersections:
                if abs(other_pt[0] - nbr[0]) < tol and abs(other_pt[1] - nbr[1]) < tol:
                    G.add_edge(node_id_map[pt], node_id_map[other_pt])
                    break

    return G

Visualize

In [22]:
def plot_intersection_graph(G, building_area, plot_area,
                            step_tag="Step4_1_IntersectionGraph",
                            show_grid=False):

    pos = nx.get_node_attributes(G, 'pos')

    fig, ax = plt.subplots(figsize=(7.5, 7.5))  # journal-friendly

    # Plot plot boundary
    x, y = plot_area.exterior.xy
    ax.plot(x, y, color='gray', linestyle='--', linewidth=1.0, label='Plot Area')

    # Plot building area
    bx, by = building_area.exterior.xy
    ax.fill(bx, by, color='lightgray', alpha=0.35, label='Building Area')

    # Plot edges
    for u, v in G.edges:
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        ax.plot([x0, x1], [y0, y1], color='blue', linewidth=0.9)

    # Plot nodes + labels
    for node, (xn, yn) in pos.items():
        ax.plot(xn, yn, marker='o', color='red', markersize=3.5)
        ax.text(xn + 0.15, yn + 0.15, str(node), color='red', fontsize=7)

    ax.set_title("Step 4.1: Numeric Grid Intersection Graph")
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.legend(frameon=False, loc="best")

    if show_grid:
        ax.grid(True, linewidth=0.3, alpha=0.35)

    # Save journal-quality output
    save_figure(fig, step_tag)

    plt.show()


In [ ]:
G_intersections = generate_intersection_graph_from_cells(grid_cells)
plot_intersection_graph(G_intersections, selected_building, spatial_data['polygon'])

plot_intersection_graph(
    G_intersections, selected_building, spatial_data['polygon'],
    step_tag=f"Step4_1_IntersectionGraph_seed{BENCHMARK_SEED}_opt{FIXED_OPTION_INDEX}"
)

Step 4.2: Graph Data Representation of Cells

Generate Cell Graphs from Existing Grids

In [24]:
import string
import networkx as nx
from shapely.geometry import Point

def get_cell_center(cell):
    minx, miny, maxx, maxy = cell.bounds
    return ((minx + maxx) / 2, (miny + maxy) / 2)

def generate_cell_graph(grid_cells):
    G = nx.Graph()
    centers = [get_cell_center(cell) for cell in grid_cells]

    # Derive cell size (assumes all grid cells are same)
    first_cell = grid_cells[0]
    minx, miny, maxx, maxy = first_cell.bounds
    cell_width = round(maxx - minx, 3)
    cell_height = round(maxy - miny, 3)
    cell_size = (cell_width, cell_height)

    # Generate alphabetical node labels
    def alpha_label(index):
        result = ""
        while True:
            index, remainder = divmod(index, 26)
            result = chr(65 + remainder) + result
            if index == 0:
                break
            index -= 1
        return result

    node_id_map = {}
    for idx, center in enumerate(centers):
        label = alpha_label(idx)
        G.add_node(label, pos=center)
        G.nodes[label]['coord'] = center         # ✅ stores center as 'coord'
        G.nodes[label]['size'] = cell_size       # ✅ stores grid cell size
        node_id_map[center] = label

    # Add edges between adjacent cells
    tolerance = 1e-3
    for c1 in centers:
        for c2 in centers:
            if c1 == c2:
                continue
            dx = abs(c1[0] - c2[0])
            dy = abs(c1[1] - c2[1])
            if (abs(dx - cell_width) < tolerance and dy < tolerance) or \
               (abs(dy - cell_height) < tolerance and dx < tolerance):
                G.add_edge(node_id_map[c1], node_id_map[c2])

    return G

Visualize Cell Graph

In [25]:
def plot_cell_graph(G_cells, building_area, plot_area,
                    step_tag="Step4_2_CellGraph",
                    show_grid=False):

    pos = nx.get_node_attributes(G_cells, 'pos')

    fig, ax = plt.subplots(figsize=(7.5, 7.5))  # journal-friendly

    # Plot plot boundary
    x, y = plot_area.exterior.xy
    ax.plot(x, y, color='gray', linestyle='--', linewidth=1.0, label='Plot Area')

    # Plot building area
    bx, by = building_area.exterior.xy
    ax.fill(bx, by, color='lightgray', alpha=0.35, label='Building Area')

    # Plot graph edges
    for u, v in G_cells.edges:
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        ax.plot([x0, x1], [y0, y1], color='green', linewidth=0.9)

    # Plot nodes with labels
    for node, (xn, yn) in pos.items():
        ax.plot(xn, yn, marker='s', color='green', markersize=4)  # green square
        ax.text(xn + 0.15, yn + 0.15, node, color='darkgreen', fontsize=7,
                ha='left', va='bottom')

    ax.set_title("Step 4.2: Grid Cells as Alphabetic Graph Nodes")
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.legend(frameon=False, loc="best")

    if show_grid:
        ax.grid(True, linewidth=0.3, alpha=0.35)

    # Save journal-quality output
    save_figure(fig, step_tag)

    plt.show()


Create Hybrid Graph

In [26]:
hybrid_graph_data = {
    "intersection_graph": G_intersections,
    "cell_graph": generate_cell_graph(grid_cells)
}

Visualize Hybrid Graph

In [ ]:
G_cells = hybrid_graph_data["cell_graph"]
plot_cell_graph(G_cells, selected_building, spatial_data['polygon'])

In [ ]:
plot_cell_graph(
    G_cells,
    selected_building,
    spatial_data['polygon'],
    step_tag=f"Step4_2_CellGraph_seed{BENCHMARK_SEED}_opt{FIXED_OPTION_INDEX}"
)


In [29]:
import matplotlib.pyplot as plt
import networkx as nx

def plot_hybrid_graph(G_intersections, G_cells, building_area, plot_area,
                      step_tag="Step4_3_HybridGraph",
                      show_grid=False):

    pos_int = nx.get_node_attributes(G_intersections, 'pos')
    pos_cells = nx.get_node_attributes(G_cells, 'pos')

    fig, ax = plt.subplots(figsize=(8, 7.5))  # journal-friendly

    # Plot plot area
    x, y = plot_area.exterior.xy
    ax.plot(x, y, color='gray', linestyle='--', linewidth=1.0, label="Plot Area")

    # Plot building area
    bx, by = building_area.exterior.xy
    ax.fill(bx, by, color='lightgrey', alpha=0.35, label="Building Area")

    # ---- G_intersections (blue edges + red nodes) ----
    for u, v in G_intersections.edges:
        x0, y0 = pos_int[u]
        x1, y1 = pos_int[v]
        ax.plot([x0, x1], [y0, y1], color='blue', linewidth=0.8)

    for node, (xn, yn) in pos_int.items():
        ax.plot(xn, yn, marker='o', color='red', markersize=3.5)
        ax.text(xn + 0.10, yn + 0.10, str(node), color='red', fontsize=7)

    # ---- G_cells (green edges + green square nodes) ----
    for u, v in G_cells.edges:
        x0, y0 = pos_cells[u]
        x1, y1 = pos_cells[v]
        ax.plot([x0, x1], [y0, y1], color='green', linewidth=0.8)

    for node, (xn, yn) in pos_cells.items():
        ax.plot(xn, yn, marker='s', color='green', markersize=4.5)
        ax.text(xn + 0.10, yn + 0.10, str(node), color='darkgreen', fontsize=7)

    ax.set_title("Step 4.3: Hybrid Graph (Intersection + Cell Nodes)")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_aspect('equal', adjustable='box')
    ax.legend(frameon=False, loc="upper right")

    if show_grid:
        ax.grid(True, linewidth=0.3, alpha=0.35)

    # Save journal-quality output
    save_figure(fig, step_tag)

    plt.show()


In [ ]:
plot_hybrid_graph(G_intersections, G_cells, selected_building, spatial_data['polygon'])


In [ ]:
plot_hybrid_graph(
    G_intersections, G_cells, selected_building, spatial_data['polygon'],
    step_tag=f"Step4_3_HybridGraph_seed{BENCHMARK_SEED}_opt{FIXED_OPTION_INDEX}"
)


***Step 5: Room Area Identification Using Flood-Fill Algorithm***

In [ ]:
# =========================
# STEP 5 (ALL SEQUENCES): generate layouts for every unique room order — with directional flood-fill INCLUDED
# =========================
import math
import copy
from collections import Counter, deque
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import networkx as nx

# ---------- utilities ----------
def base_of(name_with_uid: str) -> str:
    """'Bedroom#2' -> 'Bedroom' (map UID → base name for constraints lookup)."""
    return name_with_uid.split('#', 1)[0].strip()

def room_plan_counter(room_plan):
    """room_plan dict -> Counter of base names."""
    c = Counter()
    for k, v in room_plan.items():
        if v > 0:
            c[k] += int(v)
    return c

def count_multiset_permutations(counter: Counter) -> int:
    """Return n! / (n1!*n2!*...) for multiset described by 'counter'."""
    n = sum(counter.values())
    denom = 1
    for v in counter.values():
        denom *= math.factorial(v)
    return math.factorial(n) // denom if n > 0 else 0

def multiset_permutations_all(counter: Counter):
    """
    Deterministic generator of ALL unique permutations of a multiset.
    Yields tuples of base room names in lexicographic order by key.
    """
    items = sorted(counter.items())           # list of (key, remaining_count)
    n = sum(c for _, c in items)
    seq = [None] * n

    def backtrack(pos: int):
        if pos == n:
            yield tuple(seq)
            return
        for i, (k, c) in enumerate(items):
            if c == 0:
                continue
            items[i] = (k, c - 1)
            seq[pos] = k
            yield from backtrack(pos + 1)
            items[i] = (k, c)
    yield from backtrack(0)

def build_ordered_rooms_with_uids(room_constraints, base_sequence):
    """
    base_sequence: ('Living','Bedroom','Bedroom',...)
    Returns Room objects with unique names: 'Bedroom#1', 'Bedroom#2', ...
    NOTE: constraints are looked up by base name.
    """
    counters = Counter()
    ordered = []
    for base in base_sequence:
        counters[base] += 1
        uid_name = f"{base}#{counters[base]}"
        ordered.append(Room(uid_name, room_constraints[base]))  # Room class from Step 2
    return ordered

# ---------- sorting logic (uses node 'pos') ----------
def sort_start_nodes(G, logic):
    """
    Returns node list ordered by a geometric heuristic per 'logic'.
    logic 1: Bottom-left → X→Y
    logic 2: Bottom-right → X→Y
    logic 3: Bottom-left → Y→X
    logic 4: Bottom-right → Y→X
    """
    pos = nx.get_node_attributes(G, 'pos')
    if logic == 1:      # y asc, x asc
        return sorted(G.nodes, key=lambda n: (pos[n][1],  pos[n][0]))
    elif logic == 2:    # y asc, x desc
        return sorted(G.nodes, key=lambda n: (pos[n][1], -pos[n][0]))
    elif logic == 3:    # x asc, y asc
        return sorted(G.nodes, key=lambda n: (pos[n][0],  pos[n][1]))
    else:               # x asc, y desc (start near bottom-right along Y)
        return sorted(G.nodes, key=lambda n: (pos[n][0], -pos[n][1]))

# ---------- directional flood-fill allocation (YOUR logic) ----------
def flood_fill_with_logic(G, start_node, constraint, visited, polygon, logic):
    """
    G: cell graph with nodes having:
       - 'pos' (cx, cy)  (we set from 'coord' below if missing)
       - 'size' = (cell_w, cell_h)
    constraint: dict with keys 'area','length','width' (meters)
    polygon: shapely Polygon representing the valid building envelope
    logic: 1/2 → grow X then Y ; 3/4 → grow Y then X, with directional biases
    """
    if start_node in visited:
        return []

    pos = nx.get_node_attributes(G, 'pos')
    size = G.nodes[start_node]['size']           # (cell_w, cell_h)
    area_needed = float(constraint['area'])
    req_len     = float(constraint['length'])
    req_wid     = float(constraint['width'])
    cell_area   = size[0] * size[1]

    if logic in [1, 2]:  # X→Y
        x_count = int(math.ceil(req_len / size[0]))
        y_count = int(math.ceil(req_wid / size[1]))
        x_axis, y_axis = 0, 1
    else:                # Y→X
        x_count = int(math.ceil(req_len / size[1]))
        y_count = int(math.ceil(req_wid / size[0]))
        x_axis, y_axis = 1, 0

    def inside(n):
        x, y = pos[n]
        w, h = G.nodes[n]['size']
        cell_poly = Polygon([(x - w/2, y - h/2), (x + w/2, y - h/2), (x + w/2, y + h/2), (x - w/2, y + h/2)])
        # covers is less strict than contains; switch to contains if you require strict interior
        return polygon.covers(cell_poly)

    # Expand to reach area if the ideal grid block is short
    def expand_region(region, target_area):
        expanded = set(region)
        # frontier order bias per logic
        frontier = deque(sorted(
            region,
            key=lambda n: (pos[n][y_axis],  pos[n][x_axis]) if logic in [1, 3]
                          else (pos[n][y_axis], -pos[n][x_axis])
        ))
        while len(expanded) * cell_area < target_area and frontier:
            node = frontier.popleft()
            neighbors = list(G.neighbors(node))
            sorted_neighbors = sorted(
                neighbors,
                key=lambda n: (pos[n][y_axis],  pos[n][x_axis]) if logic in [1, 3]
                              else (pos[n][y_axis], -pos[n][x_axis])
            )
            for nbr in sorted_neighbors:
                if nbr in visited or nbr in expanded or not inside(nbr):
                    continue
                expanded.add(nbr)
                frontier.append(nbr)
                if len(expanded) * cell_area >= target_area:
                    return list(expanded)
        return list(expanded) if len(expanded) * cell_area >= area_needed else []

    # Try to grow a x_count by y_count block, then top up area if needed
    def grow():
        line = [start_node]
        current = start_node
        # grow along primary axis (x_count cells)
        for _ in range(max(0, x_count - 1)):
            neighbors = G.neighbors(current)
            candidates = [n for n in neighbors
                          if abs(pos[n][x_axis] - pos[current][x_axis] - size[x_axis]) < 1e-3
                          and abs(pos[n][y_axis] - pos[current][y_axis]) < 1e-3
                          and n not in visited and inside(n)]
            if not candidates:
                return []
            # choose by monotone increase/decrease along primary axis
            current = sorted(candidates, key=lambda n: pos[n][x_axis])[0]
            line.append(current)

        # for each in the line, grow a column/row along the secondary axis
        full = []
        for node in line:
            col = [node]
            cur2 = node
            for _ in range(max(0, y_count - 1)):
                neighbors = G.neighbors(cur2)
                candidates = [n for n in neighbors
                              if abs(pos[n][y_axis] - pos[cur2][y_axis] - size[y_axis]) < 1e-3
                              and abs(pos[n][x_axis] - pos[cur2][x_axis]) < 1e-3
                              and n not in visited and inside(n)]
                if not candidates:
                    return []
                cur2 = sorted(candidates, key=lambda n: pos[n][y_axis])[0]
                col.append(cur2)
            full.extend(col)

        full = list(set(full))
        if any(n in visited for n in full):
            return []
        # quick bbox check for length/width in either orientation
        xs = [pos[n][0] for n in full]; ys = [pos[n][1] for n in full]
        w_eff = (max(xs) - min(xs)) + size[0]
        h_eff = (max(ys) - min(ys)) + size[1]
        size_ok = (w_eff >= req_len and h_eff >= req_wid) or (w_eff >= req_wid and h_eff >= req_len)

        if len(full) * cell_area >= area_needed and size_ok:
            return full
        # otherwise, expand to reach target area (preserves directional bias)
        return expand_region(full, area_needed)

    return grow()

# ---------- layout generation for one ordered list across chosen logics ----------
def generate_initial_layouts_with_logics_UID(
    G_cells, ordered_rooms_with_uid, room_constraints, building_polygon, logics=(1, 2, 3, 4)
):
    """
    Returns: list of (allocation_dict, logic, incomplete_flag)
    - allocation_dict: { 'RoomName#k': [cell_ids...] }
    - logic:            1/2/3/4
    - incomplete_flag:  True if not all rooms placed
    """
    # Ensure 'pos' is present (we read 'pos'; Step 4 stored centers in 'coord')
    for n in G_cells.nodes:
        if 'pos' not in G_cells.nodes[n]:
            G_cells.nodes[n]['pos'] = G_cells.nodes[n]['coord']

    outputs = []
    for logic in logics:
        visited = set()
        allocation = {}
        cell_nodes = sort_start_nodes(G_cells, logic)

        for room in ordered_rooms_with_uid:
            base = base_of(room.name)
            constraint = room_constraints[base]  # lookup by base name
            placed = False

            for start in cell_nodes:
                if start in visited:
                    continue
                alloc = flood_fill_with_logic(G_cells, start, constraint, visited, building_polygon, logic)
                if alloc:
                    allocation[room.name] = alloc
                    visited.update(alloc)
                    placed = True
                    break
            # if not placed, continue; 'incomplete' will record it

        incomplete = (len(allocation) < len(ordered_rooms_with_uid))
        outputs.append((copy.deepcopy(allocation), logic, incomplete))

    return outputs

# ---------- MAIN: enumerate ALL sequences and build layouts ----------
def step5_generate_all_layouts(
    G_cells,
    room_plan,             # dict: base room -> count
    room_constraints,      # dict: base room -> constraints dict
    building_polygon,
    logics=(1, 2, 3, 4),   # choose (1,) to get exactly n! (multiset) layouts
    warn_if_huge=True
):
    """
    Returns a list of tuples:
      (layout_dict, logic, incomplete_flag, seq_index, base_sequence_tuple)
    """
    # Ensure 'pos' exists for the whole run (safety)
    for n in G_cells.nodes:
        if 'pos' not in G_cells.nodes[n]:
            G_cells.nodes[n]['pos'] = G_cells.nodes[n]['coord']

    counter = room_plan_counter(room_plan)
    total_seq = count_multiset_permutations(counter)
    est_layouts = total_seq * len(logics)

    if warn_if_huge and total_seq > 100000:
        print(f"⚠️ Warning: {total_seq} unique sequences × {len(logics)} logics = {est_layouts} layouts. "
              f"This may be slow and memory-heavy. Consider reducing rooms or logics.")
    else:
        print("Step 5 will generate ALL sequences:")
        print(f"  • unique sequences     = {total_seq}")
        print(f"  • logics per sequence  = {len(logics)}")
        print(f"  → estimated layouts    = {est_layouts}")

    outputs = []
    for seq_idx, base_seq in enumerate(multiset_permutations_all(counter)):
        ordered_rooms = build_ordered_rooms_with_uids(room_constraints, base_seq)
        layouts_for_seq = generate_initial_layouts_with_logics_UID(
            G_cells, ordered_rooms, room_constraints, building_polygon, logics=logics
        )
        for (alloc, logic, incomplete) in layouts_for_seq:
            outputs.append((alloc, logic, incomplete, seq_idx, tuple(base_seq)))
    return outputs

# ---------- Quick visual (optional) ----------
def visualize_layout_quick(layout_dict, G_cells, building_polygon, plot_polygon,
                           title="Step 5 Layout",
                           step_tag="Step5_Layout",
                           annotate_nodes=False,
                           show_grid=False):
    fig, ax = plt.subplots(figsize=(8, 8))

    # Building + plot
    ax.add_patch(
        patches.Polygon(
            np.array(building_polygon.exterior.coords),
            fill=True, color='lightgrey', alpha=0.35, label='Building Area'
        )
    )
    ax.add_patch(
        patches.Polygon(
            np.array(plot_polygon.exterior.coords),
            fill=False, edgecolor='gray', linestyle='--', linewidth=1.0, label='Plot Area'
        )
    )

    # Colors
    cmap = plt.cm.get_cmap('tab20', max(1, len(layout_dict)))
    legend_handles = []

    for idx, (room_uid, nodes) in enumerate(layout_dict.items()):
        color = cmap(idx)

        # Draw room cells
        for node in nodes:
            cx, cy = G_cells.nodes[node]['coord']
            w, h = G_cells.nodes[node]['size']
            ax.add_patch(
                patches.Rectangle(
                    (cx - w/2, cy - h/2), w, h,
                    facecolor=color, edgecolor='black', linewidth=0.35, alpha=0.65
                )
            )
            if annotate_nodes:
                ax.text(cx, cy, str(node), fontsize=5, ha='center', va='center')

        legend_handles.append(patches.Patch(facecolor=color, edgecolor='black', label=room_uid))

    # Legend
    legend_handles.extend([
        patches.Patch(facecolor='lightgrey', edgecolor='black', label='Building Area'),
        patches.Patch(facecolor='none', edgecolor='gray', linestyle='--', label='Plot Area')
    ])
    ax.legend(handles=legend_handles, loc='upper right', fontsize=7, frameon=False)

    ax.set_title(title)
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_xlim(plot_polygon.bounds[0] - 1, plot_polygon.bounds[2] + 1)
    ax.set_ylim(plot_polygon.bounds[1] - 1, plot_polygon.bounds[3] + 1)
    ax.set_aspect('equal', adjustable='box')

    if show_grid:
        ax.grid(True, linewidth=0.25, alpha=0.35)

    # Save journal-quality output
    save_figure(fig, step_tag)

    plt.show()


# ---------- Grid browser (optional) ----------
def _draw_step5_layout_with_legend(
    ax, layout_dict, G_cells, building_polygon, plot_polygon,
    title=None, annotate_nodes=False, legend_fontsize=6, show_grid=False
):
    # Building + plot
    ax.add_patch(
        patches.Polygon(
            np.array(building_polygon.exterior.coords),
            fill=True, color='lightgrey', alpha=0.35, label='Building'
        )
    )
    ax.add_patch(
        patches.Polygon(
            np.array(plot_polygon.exterior.coords),
            fill=False, edgecolor='gray', linestyle='--', linewidth=0.9, label='Plot'
        )
    )

    cmap = plt.cm.get_cmap('tab20', max(1, len(layout_dict)))
    legend_handles = []

    for idx, (room_uid, nodes) in enumerate(layout_dict.items()):
        color = cmap(idx)
        for n in nodes:
            cx, cy = G_cells.nodes[n]['coord']
            w, h = G_cells.nodes[n]['size']
            ax.add_patch(
                patches.Rectangle(
                    (cx - w/2, cy - h/2), w, h,
                    facecolor=color, edgecolor='black', linewidth=0.25, alpha=0.65
                )
            )
            if annotate_nodes:
                ax.text(cx, cy, str(n), fontsize=4.5, ha='center', va='center')

        legend_handles.append(patches.Patch(facecolor=color, edgecolor='black', label=room_uid))

    legend_handles.extend([
        patches.Patch(facecolor='lightgrey', edgecolor='black', label='Building'),
        patches.Patch(facecolor='none', edgecolor='gray', linestyle='--', label='Plot')
    ])

    ax.legend(handles=legend_handles, fontsize=legend_fontsize, loc='upper right', frameon=False)

    if title:
        ax.set_title(title, fontsize=9)

    ax.set_xlim(plot_polygon.bounds[0] - 1, plot_polygon.bounds[2] + 1)
    ax.set_ylim(plot_polygon.bounds[1] - 1, plot_polygon.bounds[3] + 1)
    ax.set_aspect('equal', adjustable='box')

    if show_grid:
        ax.grid(True, linewidth=0.25, alpha=0.3)

    ax.tick_params(labelsize=6)
    ax.set_xlabel("X", fontsize=7)
    ax.set_ylabel("Y", fontsize=7)


def visualize_step5_grid(
    all_layouts, G_cells, building_polygon, plot_polygon,
    cols=4, rows=3, start=0, annotate_nodes=False,
    export_page=True
):
    """
    all_layouts items:
      (alloc, logic, incomplete, seq_idx, base_seq_tuple)
    Shows a page of layouts as a grid and optionally exports it.
    """
    per_page = cols * rows
    end  = min(len(all_layouts), start + per_page)
    page = all_layouts[start:end]
    if not page:
        print("No layouts to display in this range.")
        return

    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3.6*rows))
    if rows*cols == 1:
        axes = np.array([[axes]])
    elif rows == 1 or cols == 1:
        axes = np.array(axes).reshape(rows, cols)

    for ax in axes.flat:
        ax.clear()

    for idx, tpl in enumerate(page):
        r = idx // cols
        c = idx % cols
        alloc, logic, incomplete, seq_idx, base_seq = tpl
        title = f"seq#{seq_idx} | L={logic} | {'INC' if incomplete else 'OK'}"
        _draw_step5_layout_with_legend(
            axes[r, c], alloc, G_cells, building_polygon, plot_polygon,
            title=title, annotate_nodes=annotate_nodes, legend_fontsize=6, show_grid=False
        )

    # Hide unused cells
    for k in range(len(page), rows*cols):
        r = k // cols
        c = k % cols
        axes[r, c].axis('off')

    fig.tight_layout()

    if export_page:
        page_tag = f"Step5_Page_start{start:04d}_cols{cols}_rows{rows}"
        save_figure(fig, page_tag)

    plt.show()


def browse_step5(
    all_layouts, G_cells, building_polygon, plot_polygon,
    cols=4, rows=3, annotate_nodes=False
):
    """
    Pager: [Enter]=next page, 'p'=prev page, number=jumps, 'q'=quit.
    """
    per_page = cols * rows
    start = 0
    n = len(all_layouts)

    while True:
        print(f"\nShowing Step-5 layouts {start}..{min(n, start+per_page)-1} of {n-1}")
        visualize_step5_grid(
            all_layouts, G_cells, building_polygon, plot_polygon,
            cols=cols, rows=rows, start=start, annotate_nodes=annotate_nodes
        )
        cmd = input("[Enter] next | p prev | q quit | #: jump start index → ").strip().lower()
        if cmd == "q":
            break
        elif cmd == "p":
            start = max(0, start - per_page)
        elif cmd == "":
            start = min(max(0, n-1), start + per_page)
        else:
            try:
                j = int(cmd)
                if 0 <= j < n:
                    start = j
                else:
                    print("Index out of range.")
            except:
                print("Unrecognized command.")

# ---------- RUN Step 5 ----------
all_step5_layouts = step5_generate_all_layouts(
    G_cells,
    room_plan=room_plan,
    room_constraints=room_constraints,
    building_polygon=selected_building,
    logics=(1, 2, 3, 4),
    warn_if_huge=True
)

print(f"\n✅ Step 5 produced {len(all_step5_layouts)} layouts "
      f"from {len(set(seq for *_ , seq in [(l,lg,inc,seq,seq_t) for (l,lg,inc,seq,seq_t) in all_step5_layouts]))} unique sequences × "
      f"{len({lg for *_ , lg, _, _, _ in all_step5_layouts})} logics.")

# (Optional) peek a few
for i, (alloc, logic, incomplete, seq_idx, seq) in enumerate(all_step5_layouts[:3]):
    tag = f"Step5_seq{seq_idx:04d}_L{logic}_{'INC' if incomplete else 'OK'}"
    visualize_layout_quick(
        alloc, G_cells, selected_building, spatial_data['polygon'],
        title=f"Step 5 • seq#{seq_idx} L={logic} {'INC' if incomplete else 'OK'}",
        step_tag=tag,
        annotate_nodes=False,
        show_grid=False
    )



In [ ]:
for i, (alloc, logic, incomplete, seq_idx, seq) in enumerate(all_step5_layouts[:3]):
    tag = f"Step5_seq{seq_idx:04d}_L{logic}_{'INC' if incomplete else 'OK'}"
    visualize_layout_quick(
        alloc, G_cells, selected_building, spatial_data['polygon'],
        title=f"Step 5 • seq#{seq_idx} L={logic} {'INC' if incomplete else 'OK'}",
        step_tag=tag
    )


In [ ]:
browse_step5(
    all_step5_layouts,
    G_cells,
    selected_building,
    spatial_data['polygon'],
    cols=4,
    rows=3,
    annotate_nodes=False
)


***Step 6: Generating Multiple Layout using Brute Force Backtracking Algorithm***

In [ ]:
# =========================
# STEP 6 (ALL SEQUENCES → ALL SHIFTED LAYOUTS) — NO PREFILTER
# =========================
import copy
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from shapely.geometry import Polygon
from shapely.ops import unary_union

# --- tiny helpers (safe to re-declare) ---
def base_of(name_with_uid: str) -> str:
    return name_with_uid.split('#', 1)[0].strip()

def build_ordered_rooms_with_uids(room_constraints, base_sequence):
    """
    base_sequence: ('Living','Dining','Bedroom',...)
    Returns unique Room objects with names 'Type#k'.
    """
    counters = Counter()
    ordered = []
    for base in base_sequence:
        counters[base] += 1
        uid_name = f"{base}#{counters[base]}"
        ordered.append(Room(uid_name, room_constraints[base]))
    return ordered


# ---------- STEP 6 core ----------
def step6_shifted_full_layouts_UID_single_sequence(
    G_cells,
    base_sequence,             # tuple/list, e.g. ('Living','Dining','Bedroom',...)
    room_constraints,
    building_polygon,
    sort_start_nodes,
    flood_fill_with_logic,
    logics=(1,2,3,4),          # enumerate these logics
    max_shifts_per_logic=None  # None = ALL start nodes; or int cap per logic
):
    """
    Explore 'shifted' starts for ONE base sequence across the given logics.
    Returns: list of (layout_dict, logic, incomplete_flag, shift_idx)
    """
    # Use the same UID scheme as Step 5
    ordered_rooms = build_ordered_rooms_with_uids(room_constraints, base_sequence)

    outputs = []
    for logic in logics:
        start_nodes = sort_start_nodes(G_cells, logic)
        if isinstance(max_shifts_per_logic, int) and max_shifts_per_logic > 0:
            start_nodes = start_nodes[:max_shifts_per_logic]

        for shift_idx, _ in enumerate(start_nodes):
            layout = {}
            visited = set()
            success = True
            current_start_idx = shift_idx  # shift the scan window

            for room in ordered_rooms:
                base = base_of(room.name)
                constraint = room_constraints[base]
                found = False

                for i in range(current_start_idx, len(start_nodes)):
                    candidate_start = start_nodes[i]
                    if candidate_start in visited:
                        continue
                    alloc = flood_fill_with_logic(
                        G_cells, candidate_start, constraint, visited, building_polygon, logic
                    )
                    if alloc:
                        layout[room.name] = alloc
                        visited.update(alloc)
                        current_start_idx = i + 1
                        found = True
                        break

                if not found:
                    success = False
                    break

            outputs.append((copy.deepcopy(layout), logic, not success, shift_idx))
    return outputs


def collect_unique_base_sequences_from_step5(all_step5_layouts):
    """
    Step 5 tuples: (alloc, logic, incomplete, seq_idx, base_seq_tuple)
    Return dict seq_idx -> base_seq_tuple
    """
    seq_index_to_base = {}
    for _, _, _, seq_idx, base_seq in all_step5_layouts:
        if seq_idx not in seq_index_to_base:
            seq_index_to_base[seq_idx] = base_seq
    return seq_index_to_base


def step6_all_sequences(
    G_cells,
    all_step5_layouts,
    room_constraints,
    building_polygon,
    sort_start_nodes,
    flood_fill_with_logic,
    logics=(1,2,3,4),
    max_shifts_per_logic=None
):
    """
    For EVERY base sequence from Step 5, run the shifted exploration.
    Returns list of: (alloc, logic, incomplete, seq_idx, base_seq_tuple, shift_idx) 
    """
    seq_index_to_base = collect_unique_base_sequences_from_step5(all_step5_layouts)
    outputs = []

    # estimate
    approx_starts = len(sort_start_nodes(G_cells, logics[0])) if logics else 0
    cap = approx_starts if max_shifts_per_logic is None else min(approx_starts, max_shifts_per_logic)
    print(f"Step 6 estimate ≈ {len(seq_index_to_base)} sequences × {len(logics)} logics × {cap} shifts/log.")

    for seq_idx, base_seq in sorted(seq_index_to_base.items()):
        seq_layouts = step6_shifted_full_layouts_UID_single_sequence(
            G_cells,
            base_sequence=base_seq,
            room_constraints=room_constraints,
            building_polygon=building_polygon,
            sort_start_nodes=sort_start_nodes,
            flood_fill_with_logic=flood_fill_with_logic,
            logics=logics,
            max_shifts_per_logic=max_shifts_per_logic
        )
        for (alloc, logic, incomplete, shift_idx) in seq_layouts:
            outputs.append((alloc, logic, incomplete, seq_idx, tuple(base_seq), shift_idx))
    return outputs


# ---------- Visual helpers ----------
def _draw_layout_on_ax(ax, layout_dict, G_cells, building_polygon, plot_polygon,
                       title=None, annotate_nodes=False, show_grid=False):
    # Building + plot boundary
    ax.add_patch(
        patches.Polygon(
            np.array(building_polygon.exterior.coords),
            fill=True, color='lightgrey', alpha=0.35
        )
    )
    ax.add_patch(
        patches.Polygon(
            np.array(plot_polygon.exterior.coords),
            fill=False, edgecolor='gray', linestyle='--', linewidth=0.9
        )
    )

    # Rooms
    cmap = plt.cm.get_cmap('tab20', max(1, len(layout_dict)))
    for idx, (room_uid, nodes) in enumerate(layout_dict.items()):
        color = cmap(idx)
        for n in nodes:
            cx, cy = G_cells.nodes[n]['coord']
            w, h   = G_cells.nodes[n]['size']
            ax.add_patch(
                patches.Rectangle(
                    (cx - w/2, cy - h/2), w, h,
                    facecolor=color, edgecolor='black', linewidth=0.25, alpha=0.65
                )
            )
            if annotate_nodes:
                ax.text(cx, cy, str(n), fontsize=4.5, ha='center', va='center')

    if title:
        ax.set_title(title, fontsize=9)

    ax.set_xlim(plot_polygon.bounds[0] - 1, plot_polygon.bounds[2] + 1)
    ax.set_ylim(plot_polygon.bounds[1] - 1, plot_polygon.bounds[3] + 1)
    ax.set_aspect('equal', adjustable='box')

    if show_grid:
        ax.grid(True, linewidth=0.25, alpha=0.3)

    ax.tick_params(labelsize=6)
    ax.set_xlabel("X", fontsize=7)
    ax.set_ylabel("Y", fontsize=7)


def _unpack_step6_item(tpl):
    """
    Works for:
      - all_step6_layouts: (alloc, logic, incomplete, seq_idx, base_seq_tuple, shift_idx)
    """
    if len(tpl) != 6:
        raise ValueError("Unexpected tuple shape in Step 6 visualizer.")
    alloc, logic, incomplete, seq_idx, base_seq, shift_idx = tpl
    return alloc, logic, incomplete, seq_idx, base_seq, shift_idx

def visualize_layouts_grid_step6(all_layouts, G_cells, building_polygon, plot_polygon,
                                 cols=4, rows=3, start=0, annotate_nodes=False,
                                 export_page=True):
    per_page = cols * rows
    end = min(len(all_layouts), start + per_page)
    page = all_layouts[start:end]
    if not page:
        print("No layouts to display in this range.")
        return

    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3.6*rows))
    if rows*cols == 1:
        axes = np.array([[axes]])
    elif rows == 1 or cols == 1:
        axes = np.array(axes).reshape(rows, cols)

    for ax in axes.flat:
        ax.clear()

    for idx, tpl in enumerate(page):
        r = idx // cols
        c = idx % cols
        alloc, logic, incomplete, seq_idx, base_seq, shift_idx = _unpack_step6_item(tpl)
        title = f"seq#{seq_idx} shift#{shift_idx} | L={logic} | {'INC' if incomplete else 'OK'}"
        _draw_layout_on_ax(
            axes[r, c], alloc, G_cells, building_polygon, plot_polygon,
            title=title, annotate_nodes=annotate_nodes, show_grid=False
        )

    # Hide unused cells
    for k in range(len(page), rows*cols):
        r = k // cols
        c = k % cols
        axes[r, c].axis('off')

    fig.tight_layout()

    # Export the current page as high-quality outputs
    if export_page:
        page_tag = f"Step6_Page_start{start:06d}_cols{cols}_rows{rows}"
        save_figure(fig, page_tag)

    plt.show()

def export_single_step6_layout(tpl, G_cells, building_polygon, plot_polygon,
                              step_tag=None, annotate_nodes=False):
    alloc, logic, incomplete, seq_idx, base_seq, shift_idx = _unpack_step6_item(tpl)

    if step_tag is None:
        step_tag = f"Step6_seq{seq_idx:04d}_shift{shift_idx:04d}_L{logic}_{'INC' if incomplete else 'OK'}"

    fig, ax = plt.subplots(figsize=(8, 8))
    title = f"Step 6 • seq#{seq_idx} shift#{shift_idx} | L={logic} | {'INC' if incomplete else 'OK'}"
    _draw_layout_on_ax(ax, alloc, G_cells, building_polygon, plot_polygon,
                       title=title, annotate_nodes=annotate_nodes, show_grid=False)

    save_figure(fig, step_tag)
    plt.show()


def browse_layouts_step6(all_layouts, G_cells, building_polygon, plot_polygon,
                         cols=4, rows=3, annotate_nodes=False):
    """
    Pager for Step 6 layouts (NO prefilter).
    """
    per_page = cols * rows
    start = 0
    n = len(all_layouts)
    while True:
        print(f"\nShowing layouts {start}..{min(n, start+per_page)-1} of {n-1}")
        visualize_layouts_grid_step6(all_layouts, G_cells, building_polygon, plot_polygon,
                                     cols=cols, rows=rows, start=start, annotate_nodes=annotate_nodes)
        cmd = input("[Enter] next | p prev | q quit | #: jump start index → ").strip().lower()
        if cmd == "q": break
        elif cmd == "p": start = max(0, start - per_page)
        elif cmd == "": start = min(n-1, start + per_page)
        else:
            try:
                j = int(cmd)
                if 0 <= j < n: start = j
                else: print("Index out of range.")
            except: print("Unrecognized command.")


# ---------- RUN STEP 6 ----------
LOGICS_6 = (1,2,3,4)       # keep in sync with Step 5 if you want 4× sequences
SHIFT_CAP = None           # set to an int (e.g., 30) to cap shifts per logic

# Unique sequence count (for reporting)
_seq_map = collect_unique_base_sequences_from_step5(all_step5_layouts)

all_step6_layouts = step6_all_sequences(
    G_cells,
    all_step5_layouts=all_step5_layouts,
    room_constraints=room_constraints,
    building_polygon=selected_building,
    sort_start_nodes=sort_start_nodes,
    flood_fill_with_logic=flood_fill_with_logic,
    logics=LOGICS_6,
    max_shifts_per_logic=SHIFT_CAP
)

print(f"✅ Step 6 produced {len(all_step6_layouts)} layouts "
      f"from {len(_seq_map)} sequences × {len(LOGICS_6)} logics × shifts.")

# Browse EVERYTHING (no Step 7 prefilter)
browse_layouts_step6(
    all_step6_layouts,
    G_cells, selected_building, spatial_data['polygon'],
    cols=4, rows=3, annotate_nodes=False
)

export_single_step6_layout(all_step6_layouts[0], G_cells, selected_building, spatial_data['polygon'])
export_single_step6_layout(all_step6_layouts[1], G_cells, selected_building, spatial_data['polygon'])
export_single_step6_layout(all_step6_layouts[2], G_cells, selected_building, spatial_data['polygon'])


***Step 7: Evaluate and Filter Layout Using Previously Defined Adjacency and Connectivity***

In [36]:
# ---- BRIDGE: build valid_layouts_full from Step 6 (no prefilter) ----
# Step 6 items: (alloc, logic, incomplete, seq_idx, base_seq, shift_idx)
valid_layouts_full = [
    (alloc, logic, [], incomplete, seq_idx, base_seq, shift_idx)
    for (alloc, logic, incomplete, seq_idx, base_seq, shift_idx) in all_step6_layouts
]


In [ ]:
# =========================
# STEP 7 (UPDATED): Rank by Adjacency + Connectivity pairs
# =========================
from shapely.geometry import Polygon
from shapely.ops import unary_union
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# --- If these helpers already exist in your notebook, you can omit duplicates ---
def base_of(name_with_uid: str) -> str:
    """'Bedroom#2' -> 'Bedroom'"""
    return name_with_uid.split('#', 1)[0].strip()

def get_room_polygon(G_cells, node_list):
    cells = []
    for n in node_list:
        cx, cy = G_cells.nodes[n]['coord']
        w, h   = G_cells.nodes[n]['size']
        cells.append(Polygon([
            (cx-w/2, cy-h/2), (cx+w/2, cy-h/2),
            (cx+w/2, cy+h/2), (cx-w/2, cy+h/2)
        ]))
    return unary_union(cells)

def rooms_are_adjacent(poly1, poly2, tol=0.05):
    return poly1.touches(poly2) and poly1.distance(poly2) < tol
# -------------------------------------------------------------------------------

def evaluate_layout_score_UID(layout_uid_dict, adjacency_matrix, connectivity_matrix, G_cells,
                              thresholds={1: 6.0, 2: 12.0, 3: 20.0}):
    """
    For a single layout with UID keys (e.g., 'Bedroom#1'), compute:
      - pairs_total: # of required base pairs (Adj==1 or Conn>0) across *UID pairs* (unordered)
      - hits_adj / hits_conn / hits_both
      - relationships: [(uid1, uid2, status)] where status ∈ {'Adj','Conn','Adj+Conn'}
      - is_valid_or: True if every required pair satisfied by (Adj OR Conn)
    Notes:
      * We count unordered UID pairs once (no double counting r1,r2 vs r2,r1).
      * Connectivity level is taken as max(level(i,j), level(j,i)) to be robust.
    """
    # Precompute polygons
    room_polys = {uid: get_room_polygon(G_cells, nodes) for uid, nodes in layout_uid_dict.items()}

    uids = list(layout_uid_dict.keys())
    relationships = []
    pairs_total = 0
    hits_adj = 0
    hits_conn = 0
    hits_both = 0
    valid_or = True

    seen = set()  # to avoid double counting UID pairs
    for i in range(len(uids)):
        for j in range(i+1, len(uids)):
            u1, u2 = uids[i], uids[j]
            key = frozenset((u1, u2))
            if key in seen:
                continue
            seen.add(key)

            b1, b2 = base_of(u1), base_of(u2)

            # Look up requirements; guard missing keys
            adj_req = 0
            conn_req = 0
            if (adjacency_matrix is not None and b1 in adjacency_matrix.index and b2 in adjacency_matrix.columns):
                adj_req = int(adjacency_matrix.loc[b1, b2])
            if (connectivity_matrix is not None and b1 in connectivity_matrix.index and b2 in connectivity_matrix.columns):
                conn_req = max(int(connectivity_matrix.loc[b1, b2]),
                               int(connectivity_matrix.loc[b2, b1]))  # symmetric demand

            if adj_req == 0 and conn_req == 0:
                continue  # not a required pair

            pairs_total += 1

            # Evaluate
            poly1, poly2 = room_polys[u1], room_polys[u2]

            pass_adj = True if adj_req == 0 else rooms_are_adjacent(poly1, poly2)

            pass_conn = True
            if conn_req > 0:
                d = poly1.centroid.distance(poly2.centroid)
                th = thresholds.get(conn_req, 999.0)
                pass_conn = (d <= th)

            # Validate OR condition
            ok = (pass_adj or pass_conn)
            if not ok:
                valid_or = False  # this layout would fail the old filter

            # Tally + relationship label (only add satisfied ones for drawing)
            if pass_adj and pass_conn:
                hits_both += 1; hits_adj += 1; hits_conn += 1
                relationships.append((u1, u2, "Adj+Conn"))
            elif pass_adj:
                hits_adj += 1
                relationships.append((u1, u2, "Adj"))
            elif pass_conn:
                hits_conn += 1
                relationships.append((u1, u2, "Conn"))
            # else: required but failed — do not append to relationships

    return {
        "pairs_total": pairs_total,
        "hits_adj": hits_adj,
        "hits_conn": hits_conn,
        "hits_both": hits_both,
        "relationships": relationships,
        "is_valid_or": valid_or
    }

def rank_layouts_by_adj_conn(valid_layouts_full, adjacency_matrix, connectivity_matrix, G_cells,
                             w_both=2.0, w_adj_only=1.0, w_conn_only=1.0, top_k=None):
    """
    valid_layouts_full items are:
      (alloc, logic, relationships(from Step 6), incomplete, seq_idx, base_seq, shift_idx)

    Returns:
      ranked_full : list of tuples
        (alloc, logic, relationships_scored, incomplete, seq_idx, base_seq, shift_idx, score_info)
      ranked_legacy : list of (alloc, logic, relationships_scored, incomplete)  # for downstream
    """
    ranked = []
    for (alloc, logic, _rels6, incomplete, seq_idx, base_seq, shift_idx) in valid_layouts_full:
        info = evaluate_layout_score_UID(alloc, adjacency_matrix, connectivity_matrix, G_cells)
        # Split adj-only / conn-only counts
        adj_only = info["hits_adj"] - info["hits_both"]
        conn_only = info["hits_conn"] - info["hits_both"]
        score = (w_both * info["hits_both"] +
                 w_adj_only * adj_only +
                 w_conn_only * conn_only)
        ranked.append((
            alloc, logic, info["relationships"], incomplete, seq_idx, base_seq, shift_idx,
            {
                "score": score,
                "pairs_total": info["pairs_total"],
                "hits_both": info["hits_both"],
                "hits_adj_only": adj_only,
                "hits_conn_only": conn_only
            }
        ))

    # Sort: score desc, then more both, then adj, then conn, then fewer incompletes
    ranked.sort(key=lambda t: (
        -t[7]["score"],
        -t[7]["hits_both"],
        -t[7]["hits_adj_only"],
        -t[7]["hits_conn_only"],
        t[3]  # incomplete False(0) < True(1)
    ))

    if top_k is not None and top_k > 0:
        ranked = ranked[:top_k]

    # Legacy 4-tuple for your existing plotting code (now sorted by score)
    ranked_legacy = [(a, l, rels, inc) for (a, l, rels, inc, *_rest) in ranked]
    return ranked, ranked_legacy

# -------- Browser with score in title (works on 'ranked_full') --------
def _draw_layout_on_ax(ax, layout_dict, G_cells, building_polygon, plot_polygon,
                       title=None, annotate_nodes=False, show_grid=False):

    # Building + plot boundary
    ax.add_patch(
        patches.Polygon(
            np.array(building_polygon.exterior.coords),
            fill=True, color='lightgrey', alpha=0.35
        )
    )
    ax.add_patch(
        patches.Polygon(
            np.array(plot_polygon.exterior.coords),
            fill=False, edgecolor='gray', linestyle='--', linewidth=0.9
        )
    )

    # Rooms
    cmap = plt.cm.get_cmap('tab20', max(1, len(layout_dict)))
    for idx, (room_uid, nodes) in enumerate(layout_dict.items()):
        color = cmap(idx)
        for n in nodes:
            cx, cy = G_cells.nodes[n]['coord']
            w, h   = G_cells.nodes[n]['size']
            ax.add_patch(
                patches.Rectangle(
                    (cx - w/2, cy - h/2), w, h,
                    facecolor=color, edgecolor='black', linewidth=0.25, alpha=0.65
                )
            )
            if annotate_nodes:
                ax.text(cx, cy, str(n), fontsize=4.5, ha='center', va='center')

    if title:
        ax.set_title(title, fontsize=8.5)

    ax.set_xlim(plot_polygon.bounds[0] - 1, plot_polygon.bounds[2] + 1)
    ax.set_ylim(plot_polygon.bounds[1] - 1, plot_polygon.bounds[3] + 1)
    ax.set_aspect('equal', adjustable='box')

    if show_grid:
        ax.grid(True, linewidth=0.25, alpha=0.3)

    ax.tick_params(labelsize=6)
    ax.set_xlabel("X", fontsize=7)
    ax.set_ylabel("Y", fontsize=7)


def _draw_relationship_lines(ax, layout_dict, relationships, G_cells, add_legend=False):
    legend_used = {"Adj+Conn": False, "Adj": False, "Conn": False}

    for (u1, u2, status) in relationships:
        poly1 = get_room_polygon(G_cells, layout_dict[u1])
        poly2 = get_room_polygon(G_cells, layout_dict[u2])
        c1, c2 = poly1.centroid, poly2.centroid

        if status == "Adj+Conn":
            color, lab = "black", "Adj+Conn"
        elif status == "Adj":
            color, lab = "green", "Adj"
        else:
            color, lab = "blue", "Conn"

        # label only once per type for legend
        lbl = lab if (add_legend and not legend_used[lab]) else None
        legend_used[lab] = True

        ax.plot([c1.x, c2.x], [c1.y, c2.y],
                linestyle='--', linewidth=0.9, color=color, label=lbl)

    if add_legend:
        ax.legend(loc="lower left", fontsize=6, frameon=False)


def visualize_ranked_grid_step7(ranked_full, G_cells, building_polygon, plot_polygon,
                                cols=3, rows=3, start=0, annotate_nodes=False,
                                export_page=True):
    per_page = cols * rows
    end = min(len(ranked_full), start + per_page)
    page = ranked_full[start:end]
    if not page:
        print("No layouts to display in this range.")
        return

    fig, axes = plt.subplots(rows, cols, figsize=(4.2*cols, 3.8*rows))
    if rows*cols == 1:
        axes = np.array([[axes]])
    elif rows == 1 or cols == 1:
        axes = np.array(axes).reshape(rows, cols)

    for ax in axes.flat:
        ax.clear()

    for idx, tpl in enumerate(page):
        r = idx // cols
        c = idx % cols
        alloc, logic, rels, incomplete, seq_idx, base_seq, shift_idx, info = tpl

        t = (f"score={info['score']:.1f} | both={info['hits_both']} "
             f"| adj={info['hits_adj_only']} | conn={info['hits_conn_only']} "
             f"| pairs={info['pairs_total']} | seq#{seq_idx} shift#{shift_idx} | L={logic} "
             f"| {'INC' if incomplete else 'OK'}")

        _draw_layout_on_ax(
            axes[r, c], alloc, G_cells, building_polygon, plot_polygon,
            title=t, annotate_nodes=annotate_nodes, show_grid=False
        )

        # Add legend only on the first subplot to avoid clutter
        _draw_relationship_lines(
            axes[r, c], alloc, rels, G_cells,
            add_legend=(idx == 0)
        )

    # Hide unused axes
    for k in range(len(page), rows*cols):
        r = k // cols
        c = k % cols
        axes[r, c].axis('off')

    fig.tight_layout()

    if export_page:
        page_tag = f"Step7_Page_start{start:06d}_cols{cols}_rows{rows}"
        save_figure(fig, page_tag)

    plt.show()

def export_single_step7_layout(tpl, G_cells, building_polygon, plot_polygon,
                              step_tag=None, annotate_nodes=False):
    alloc, logic, rels, incomplete, seq_idx, base_seq, shift_idx, info = tpl

    if step_tag is None:
        step_tag = f"Step7_Best_seq{seq_idx:04d}_shift{shift_idx:04d}_L{logic}_score{info['score']:.1f}"

    fig, ax = plt.subplots(figsize=(8, 8))
    title = (f"score={info['score']:.1f} | both={info['hits_both']} | adj={info['hits_adj_only']} "
             f"| conn={info['hits_conn_only']} | seq#{seq_idx} shift#{shift_idx} | L={logic} "
             f"| {'INC' if incomplete else 'OK'}")

    _draw_layout_on_ax(ax, alloc, G_cells, building_polygon, plot_polygon,
                       title=title, annotate_nodes=annotate_nodes, show_grid=False)
    _draw_relationship_lines(ax, alloc, rels, G_cells, add_legend=True)

    save_figure(fig, step_tag)
    plt.show()


def browse_ranked_step7(ranked_full, G_cells, building_polygon, plot_polygon,
                        cols=4, rows=3, annotate_nodes=False):
    per_page = cols * rows
    start = 0
    n = len(ranked_full)
    while True:
        print(f"\nShowing ranked layouts {start}..{min(n, start+per_page)-1} of {n-1}")
        visualize_ranked_grid_step7(ranked_full, G_cells, building_polygon, plot_polygon,
                                    cols=cols, rows=rows, start=start, annotate_nodes=annotate_nodes)
        cmd = input("[Enter] next | p prev | q quit | #: jump start index → ").strip().lower()
        if cmd == "q":
            break
        elif cmd == "p":
            start = max(0, start - per_page)
        elif cmd == "":
            start = min(n-1, start + per_page)
        else:
            try:
                j = int(cmd)
                if 0 <= j < n:
                    start = j
                else:
                    print("Index out of range.")
            except:
                print("Unrecognized command.")

# ===== Run Step 7 on Step 6 outputs =====
# Use the Step-6 filtered list with metadata if you built it there:
#   valid_layouts_full = [(alloc, logic, relationships, incomplete, seq_idx, base_seq, shift_idx), ...]
# If you DON'T have valid_layouts_full, build it from all_step6_layouts before calling this.

ranked_full, ranked_legacy = rank_layouts_by_adj_conn(
    valid_layouts_full,
    adjacency_matrix, connectivity_matrix, G_cells,
    w_both=2.0, w_adj_only=1.0, w_conn_only=1.0,   # tweak weights if you want
    top_k=None                                      # or set an int to keep only top N
)

print(f"✅ Step 7 scored {len(ranked_full)} layouts. Top score = {ranked_full[0][7]['score'] if ranked_full else 'n/a'}")

# Keep variables you asked for:
best_layouts_full = ranked_full                    # rich tuples (with score & metadata)
best_layouts_legacy = ranked_legacy                # (alloc, logic, relationships, incomplete)

# Browse like Step 5/6, with score annotation:
browse_ranked_step7(best_layouts_full, G_cells, selected_building, spatial_data['polygon'],
                    cols=4, rows=3, annotate_nodes=False)

export_single_step7_layout(best_layouts_full[0], G_cells, selected_building, spatial_data['polygon'])
export_single_step7_layout(best_layouts_full[1], G_cells, selected_building, spatial_data['polygon'])
export_single_step7_layout(best_layouts_full[2], G_cells, selected_building, spatial_data['polygon'])


In [ ]:
# =========================
# Step 7 — keep only highest-scoring layouts (+ browser)
# =========================

# --- tiny helper to dedupe identical allocations ---
def _layout_fingerprint_uid(alloc_dict):
    # stable, hashable signature from UID->node list mapping
    items = []
    for k, v in sorted(alloc_dict.items()):
        items.append((k, tuple(sorted(v))))
    return tuple(items)

def select_top_layouts(
    ranked_full,
    top_k=20,                 # keep the best N overall
    min_score=None,           # or enforce a minimum score
    one_per_sequence=False,   # True → best one per sequence id
    dedupe=True               # drop duplicate allocations across shifts/logics
):
    """
    ranked_full items:
      (alloc, logic, relationships, incomplete, seq_idx, base_seq, shift_idx, score_info)
    Returns:
      top_full   : same tuple + score_info
      top_legacy : [(alloc, logic, relationships, incomplete), ...]  (for Step 8)
    """
    out = []
    seen = set()
    best_per_seq = {}

    # already sorted by score desc in your Step 7
    for tpl in ranked_full:
        alloc, logic, rels, incomplete, seq_idx, base_seq, shift_idx, info = tpl

        # score/threshold filter
        if (min_score is not None) and (info["score"] < float(min_score)):
            continue

        # one-per-sequence policy
        if one_per_sequence:
            if seq_idx in best_per_seq:
                continue
            best_per_seq[seq_idx] = True

        # dedupe identical allocations
        if dedupe:
            fp = _layout_fingerprint_uid(alloc)
            if fp in seen:
                continue
            seen.add(fp)

        out.append(tpl)
        if (top_k is not None) and (len(out) >= int(top_k)):
            break

    # legacy 4-tuple shape for downstream code
    out_legacy = [(a, l, rels, inc) for (a, l, rels, inc, *_rest) in out]
    return out, out_legacy

# -------- pick your policy here --------
# Option A: top 30 overall (deduped)
top_full, top_legacy = select_top_layouts(ranked_full, top_k=30, min_score=None,
                                          one_per_sequence=False, dedupe=True)

# Option B: one best per sequence id (nice variety)
# top_full, top_legacy = select_top_layouts(ranked_full, top_k=None, one_per_sequence=True, dedupe=True)

print(f"Kept {len(top_full)} highest-scoring layouts.")

# Make these the canonical outputs for the next step:
best_layouts_full   = top_full                     # rich tuples with score_info
best_layouts_legacy = top_legacy                   # 4-tuples (alloc, logic, relationships, incomplete)

# If your Step 8 expects a variable literally named 'valid_layouts', set it now:
valid_layouts = best_layouts_legacy

# -------- browse ONLY the kept layouts --------
browse_ranked_step7(best_layouts_full, G_cells, selected_building, spatial_data['polygon'],
                    cols=4, rows=3, annotate_nodes=False)

# (Optional) quick text summary
for i, tpl in enumerate(best_layouts_full[:10]):
    _, logic, rels, inc, seq_idx, base_seq, shift_idx, info = tpl
    print(f"#{i:02d} score={info['score']:.1f} | both={info['hits_both']} "
          f"| adj={info['hits_adj_only']} | conn={info['hits_conn_only']} "
          f"| pairs={info['pairs_total']} | seq#{seq_idx} shift#{shift_idx} | L={logic} | {'INC' if inc else 'OK'}")


Evaluate Layouts

***Step 8: Area Maximization***

In [41]:
# Expect 7-tuple items in best_layouts_full
assert best_layouts_full and len(best_layouts_full[0]) == 8
a,l,rels,inc,seq_idx,base_seq,shift_idx,info = best_layouts_full[0]
print("Step 7 → OK:", type(a) is dict, "logic:", l, "seq:", seq_idx, "shift:", shift_idx, "score:", info.get("score"))


Step 7 → OK: True logic: 1 seq: 2 shift: 9 score: 29.0


In [42]:
# =========================
# STEP 8 (UPDATED): Bulk room expansion from Step-7 best layouts
# =========================
import copy
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# We reuse helpers defined earlier:
# - base_of(name_with_uid)                # 'Bedroom#2' -> 'Bedroom'
# - sort_start_nodes(G, logic)
# - flood_fill_with_logic(G, start_node, constraint, visited, polygon, logic)
# - evaluate_layout_score_UID(layout_uid_dict, adjacency_matrix, connectivity_matrix, G_cells)
# - get_room_polygon(G_cells, node_list)

# ---------- core: expand ONE room (UID) inside ONE layout ----------
def expand_room_panelized_uid(
    G_cells,
    layout_uid_dict,         # {'Living#1':[nodes...], 'Bedroom#1':[...], ...}
    logic,                   # 1..4 (same logic used to generate the layout)
    room_uid,                # e.g. 'Living#1'
    max_extra_cells,         # int: try +1..+N
    room_constraints,        # dict keyed by BASE names ('Living', 'Bedroom', ...)
    building_polygon,
    keep_other_rooms_fixed=True
):
    """
    Returns a list of tuples:
      (new_layout_dict, logic, extra_cells, expanded_room_uid, incomplete_flag)
    """
    # quick geometry info
    any_node = next(iter(G_cells.nodes))
    cw, ch = G_cells.nodes[any_node]['size']
    cell_area = cw * ch

    original_layout = copy.deepcopy(layout_uid_dict)
    if room_uid not in original_layout:
        return []

    original_nodes = list(original_layout[room_uid])
    base = base_of(room_uid)

    variants = []

    for extra in range(1, int(max_extra_cells) + 1):
        # bump the area target only for the expanded room's BASE type
        constraints_copy = copy.deepcopy(room_constraints)
        constraints_copy[base]['area'] += extra * cell_area

        if keep_other_rooms_fixed:
            # keep others in place; only try to grow target room
            fixed = {k: v for k, v in original_layout.items() if k != room_uid}
            visited = set(n for nodes in fixed.values() for n in nodes)
            out_layout = copy.deepcopy(fixed)

            placed = False
            for start in original_nodes:
                alloc = flood_fill_with_logic(
                    G_cells, start,
                    constraints_copy[base], visited, building_polygon, logic
                )
                if alloc:
                    out_layout[room_uid] = alloc
                    placed = True
                    break

            if not placed:
                # failed variant; skip to next +cells
                continue

            incomplete = False  # we kept others, so it's complete if target placed

        else:
            # free mode: place target from its old cells, then re-place the others
            out_layout = {}
            visited = set()

            # 1) place the expanded target
            placed = False
            for start in original_nodes:
                alloc = flood_fill_with_logic(
                    G_cells, start,
                    constraints_copy[base], visited, building_polygon, logic
                )
                if alloc:
                    out_layout[room_uid] = alloc
                    visited.update(alloc)
                    placed = True
                    break
            if not placed:
                continue  # can't place target → skip

            # 2) re-place the rest using their BASE constraints
            start_nodes = sort_start_nodes(G_cells, logic)
            incomplete = False
            for uid, nodes in original_layout.items():
                if uid == room_uid:
                    continue
                b = base_of(uid)
                req = constraints_copy[b]
                ok = False
                for s in start_nodes:
                    if s in visited:
                        continue
                    alloc = flood_fill_with_logic(G_cells, s, req, visited, building_polygon, logic)
                    if alloc:
                        out_layout[uid] = alloc
                        visited.update(alloc)
                        ok = True
                        break
                if not ok:
                    incomplete = True
                    # we keep going; variant can still be useful downstream

        variants.append((
            copy.deepcopy(out_layout), logic, extra, room_uid, incomplete
        ))

    return variants


# ---------- driver: expand SELECTED base room types across MANY layouts ----------
def step8_bulk_expand_from_ranked(
    best_layouts_full,           # from Step 7 (ranked_full)
    G_cells,
    room_constraints,
    building_polygon,
    adjacency_matrix,
    connectivity_matrix,
    selected_bases,             # list of BASE names to expand, e.g. ['Living','Dining'] or ['all']
    max_extra_cells=2,
    keep_other_rooms_fixed=True,
    keep_only_valid=True,       # True => keep only (Adj OR Conn)-valid variants
    cap_per_layout=None         # int: optional cap of variants per source layout
):
    """
    best_layouts_full items:
      (alloc, logic, relationships, incomplete, seq_idx, base_seq, shift_idx, score_info)

    Returns:
      step8_expanded_full  : list of
         (alloc_new, logic, relationships_new, incomplete, seq_idx, base_seq, shift_idx,
          {'expanded_room_uid': uid, 'extra_cells': k,
           'source_index': src_i, 'source_score': src_score})
      valid_layouts_step8  : legacy 4-tuples [(alloc, logic, relationships, incomplete), ...]
                             (good for your Step 9/10 pipeline or GA init)
    """
    expanded_full = []
    add_count = 0

    # normalize base selection
    if isinstance(selected_bases, str):
        selected_bases = [selected_bases]
    selected_bases = [b.strip() for b in selected_bases]
    select_all = ("all" in [b.lower() for b in selected_bases])

    for src_i, tpl in enumerate(best_layouts_full):
        alloc, logic, _rels, incomplete_src, seq_idx, base_seq, shift_idx, score_info = tpl
        src_score = score_info.get("score", 0.0)

        # decide which UIDs to expand
        uids = list(alloc.keys())
        uids_to_expand = []
        for uid in uids:
            b = base_of(uid)
            if select_all or (b in selected_bases):
                uids_to_expand.append(uid)

        per_layout_added = 0
        for uid in uids_to_expand:
            variants = expand_room_panelized_uid(
                G_cells, alloc, logic, uid, max_extra_cells,
                room_constraints, building_polygon,
                keep_other_rooms_fixed=keep_other_rooms_fixed
            )

            for (alloc_new, logic_new, extra, uid_exp, incomplete_new) in variants:
                # evaluate adj/conn for the new layout
                info = evaluate_layout_score_UID(alloc_new, adjacency_matrix, connectivity_matrix, G_cells)
                if keep_only_valid and (not info["is_valid_or"]):
                    continue

                expanded_full.append((
                    alloc_new, logic_new, info["relationships"], incomplete_new,
                    seq_idx, base_seq, shift_idx,
                    {
                        "expanded_room_uid": uid_exp,
                        "extra_cells": int(extra),
                        "source_index": int(src_i),
                        "source_score": float(src_score),
                        "pairs_total": int(info["pairs_total"]),
                        "hits_both": int(info["hits_both"]),
                        "hits_adj_only": int(info["hits_adj"] - info["hits_both"]),
                        "hits_conn_only": int(info["hits_conn"] - info["hits_both"]),
                    }
                ))
                add_count += 1
                per_layout_added += 1
                if isinstance(cap_per_layout, int) and per_layout_added >= cap_per_layout:
                    break

    # legacy shape for downstream steps
    valid_layouts_step8 = [(a, l, rels, inc) for (a, l, rels, inc, *_rest) in expanded_full]

    print(f"✅ Step 8 produced {add_count} expanded layouts from {len(best_layouts_full)} source layouts.")
    return expanded_full, valid_layouts_step8


# ---------- simple browser for Step-8 variants ----------
def _draw_layout_on_ax(ax, layout_dict, G_cells, building_polygon, plot_polygon,
                       title=None, annotate_nodes=False, show_grid=False):

    ax.add_patch(
        patches.Polygon(
            np.array(building_polygon.exterior.coords),
            fill=True, color='lightgrey', alpha=0.35
        )
    )
    ax.add_patch(
        patches.Polygon(
            np.array(plot_polygon.exterior.coords),
            fill=False, edgecolor='gray', linestyle='--', linewidth=0.9
        )
    )

    cmap = plt.cm.get_cmap('tab20', max(1, len(layout_dict)))
    for idx, (room_uid, nodes) in enumerate(layout_dict.items()):
        color = cmap(idx)
        for n in nodes:
            cx, cy = G_cells.nodes[n]['coord']
            w, h   = G_cells.nodes[n]['size']
            ax.add_patch(
                patches.Rectangle(
                    (cx - w/2, cy - h/2), w, h,
                    facecolor=color, edgecolor='black', linewidth=0.25, alpha=0.65
                )
            )
            if annotate_nodes:
                ax.text(cx, cy, str(n), fontsize=4.5, ha='center', va='center')

    if title:
        ax.set_title(title, fontsize=8.5)

    ax.set_xlim(plot_polygon.bounds[0] - 1, plot_polygon.bounds[2] + 1)
    ax.set_ylim(plot_polygon.bounds[1] - 1, plot_polygon.bounds[3] + 1)
    ax.set_aspect('equal', adjustable='box')

    if show_grid:
        ax.grid(True, linewidth=0.25, alpha=0.3)

    ax.tick_params(labelsize=6)
    ax.set_xlabel("X", fontsize=7)
    ax.set_ylabel("Y", fontsize=7)


def visualize_step8_grid(expanded_full, G_cells, building_polygon, plot_polygon,
                         cols=3, rows=3, start=0, annotate_nodes=False,
                         export_page=True):
    per_page = cols * rows
    end = min(len(expanded_full), start + per_page)
    page = expanded_full[start:end]
    if not page:
        print("No layouts to display in this range.")
        return

    fig, axes = plt.subplots(rows, cols, figsize=(4.2*cols, 3.8*rows))
    if rows*cols == 1:
        axes = np.array([[axes]])
    elif rows == 1 or cols == 1:
        axes = np.array(axes).reshape(rows, cols)

    for ax in axes.flat:
        ax.clear()

    for idx, tpl in enumerate(page):
        r = idx // cols
        c = idx % cols
        alloc, logic, rels, inc, seq_idx, base_seq, shift_idx, meta = tpl

        tt = (f"+{meta['extra_cells']} {meta['expanded_room_uid']} | "
              f"pairs={meta['pairs_total']} | both={meta['hits_both']} "
              f"| adj={meta['hits_adj_only']} | conn={meta['hits_conn_only']} | "
              f"src#{meta['source_index']} score={meta['source_score']:.1f} | "
              f"seq#{seq_idx} shift#{shift_idx} | L={logic} | {'INC' if inc else 'OK'}")

        _draw_layout_on_ax(
            axes[r, c], alloc, G_cells, building_polygon, plot_polygon,
            title=tt, annotate_nodes=annotate_nodes, show_grid=False
        )

    # Hide unused axes
    for k in range(len(page), rows*cols):
        r = k // cols
        c = k % cols
        axes[r, c].axis('off')

    fig.tight_layout()

    if export_page:
        page_tag = f"Step8_Page_start{start:06d}_cols{cols}_rows{rows}"
        save_figure(fig, page_tag)

    plt.show()

def export_single_step8_layout(tpl, G_cells, building_polygon, plot_polygon,
                              step_tag=None, annotate_nodes=False):
    alloc, logic, rels, inc, seq_idx, base_seq, shift_idx, meta = tpl

    if step_tag is None:
        step_tag = (f"Step8_seq{seq_idx:04d}_shift{shift_idx:04d}_L{logic}_"
                    f"plus{meta['extra_cells']}_{meta['expanded_room_uid'].replace('#','_')}")

    fig, ax = plt.subplots(figsize=(8, 8))
    title = (f"Expanded: +{meta['extra_cells']} cells to {meta['expanded_room_uid']} | "
             f"src#{meta['source_index']} score={meta['source_score']:.1f} | "
             f"seq#{seq_idx} shift#{shift_idx} | L={logic} | {'INC' if inc else 'OK'}")

    _draw_layout_on_ax(ax, alloc, G_cells, building_polygon, plot_polygon,
                       title=title, annotate_nodes=annotate_nodes, show_grid=False)

    save_figure(fig, step_tag)
    plt.show()



def browse_step8(expanded_full, G_cells, building_polygon, plot_polygon,
                 cols=4, rows=3, annotate_nodes=False):
    per_page = cols * rows
    start = 0
    n = len(expanded_full)
    while True:
        print(f"\nStep 8 — showing variants {start}..{min(n, start+per_page)-1} of {n-1}")
        visualize_step8_grid(expanded_full, G_cells, building_polygon, plot_polygon,
                             cols=cols, rows=rows, start=start, annotate_nodes=annotate_nodes)
        cmd = input("[Enter] next | p prev | q quit | #: jump start index → ").strip().lower()
        if cmd == "q":
            break
        elif cmd == "p":
            start = max(0, start - per_page)
        elif cmd == "":
            start = min(n-1, start + per_page)
        else:
            try:
                j = int(cmd)
                if 0 <= j < n:
                    start = j
                else:
                    print("Index out of range.")
            except:
                print("Unrecognized command.")


# ---------- quick interactive front door ----------
def step8_interactive(
    best_layouts_full,
    G_cells,
    room_constraints,
    building_polygon,
    adjacency_matrix,
    connectivity_matrix,
    spatial_data,
    default_bases="all",
    default_max_extra=2,
    default_keep_fixed="y",
    keep_only_valid=True,
    cap_per_layout=None
):
    # discover available BASE types across the first few layouts
    bases = set()
    for (alloc, *_rest) in [tpl for tpl in best_layouts_full[: min(10, len(best_layouts_full))]]:
        for uid in alloc.keys():
            bases.add(base_of(uid))
    print("\nAvailable base room types:", ", ".join(sorted(bases)))

    raw = input(f"Which room types to expand? (comma or 'all') [default: {default_bases}]: ").strip()
    selected = default_bases if raw == "" else raw
    if selected.lower().strip() == "all":
        selected_bases = ["all"]
    else:
        selected_bases = [s.strip() for s in selected.split(",") if s.strip()]

    rawN = input(f"Max extra cells to try per room [default: {default_max_extra}]: ").strip()
    max_extra = default_max_extra if rawN == "" else int(rawN)

    rawK = input(f"Keep other rooms fixed? (y/n) [default: {default_keep_fixed}]: ").strip().lower()
    keep_fixed = (rawK == "" and default_keep_fixed.lower() == "y") or (rawK == "y")

    expanded_full, valid_layouts_step8 = step8_bulk_expand_from_ranked(
        best_layouts_full,
        G_cells, room_constraints, building_polygon,
        adjacency_matrix, connectivity_matrix,
        selected_bases=selected_bases,
        max_extra_cells=max_extra,
        keep_other_rooms_fixed=keep_fixed,
        keep_only_valid=keep_only_valid,
        cap_per_layout=cap_per_layout
    )

    print(f"\nExpanded variants kept: {len(expanded_full)}")
    if expanded_full:
        browse_step8(expanded_full, G_cells, building_polygon, spatial_data['polygon'],
                     cols=4, rows=3, annotate_nodes=False)

    return expanded_full, valid_layouts_step8


# ===== Run Step 8 (two ways) =====
# 1) Interactive (recommended for first run)
# step8_expanded_full, valid_layouts_step8 = step8_interactive(
#     best_layouts_full, G_cells, room_constraints, selected_building,
#     adjacency_matrix, connectivity_matrix, spatial_data,
#     default_bases="all", default_max_extra=2, default_keep_fixed="y",
#     keep_only_valid=True, cap_per_layout=None
# )

# 2) Programmatic (no prompts). Example:
# step8_expanded_full, valid_layouts_step8 = step8_bulk_expand_from_ranked(
#     best_layouts_full,
#     G_cells, room_constraints, selected_building,
#     adjacency_matrix, connectivity_matrix,
#     selected_bases=["Living","Dining","Kitchen"],  # or ["all"] 
#     max_extra_cells=2,
#     keep_other_rooms_fixed=True,
#     keep_only_valid=True,
#     cap_per_layout=None
# )

# Make these available for your GA step as the "initial population":
#   - 'valid_layouts_step8' already has the legacy 4-tuple shape:
#         (layout_dict, logic, relationships, incomplete)
#   - If your existing GA channel expects a variable literally named 'valid_layouts',
#     you can alias it:
# valid_layouts = valid_layouts_step8


In [ ]:
# Runs Step 8 and opens a pager to browse the new variants
step8_expanded_full, valid_layouts_step8 = step8_interactive(
    best_layouts_full,                 # from Step 7
    G_cells, room_constraints,         # from earlier steps
    selected_building,                 # building polygon
    adjacency_matrix, connectivity_matrix,
    spatial_data,                      # for plotting the plot boundary
    default_bases="all",               # or e.g. "Living,Dining"
    default_max_extra=2,               # try +1..+2 cells per chosen room
    default_keep_fixed="y",            # keep other rooms fixed (y/n)
    keep_only_valid=True,              # keep only layouts passing (Adj OR Conn)
    cap_per_layout=None                # or an int to limit variants per source layout
)

# If the next steps expect a variable literally named 'valid_layouts':
valid_layouts = valid_layouts_step8

export_single_step8_layout(step8_expanded_full[0], G_cells, selected_building, spatial_data['polygon'])
export_single_step8_layout(step8_expanded_full[1], G_cells, selected_building, spatial_data['polygon'])
export_single_step8_layout(step8_expanded_full[2], G_cells, selected_building, spatial_data['polygon'])

***Step 9: Panelization***

In [ ]:
# ================================
# Panelization on valid_layouts_step8 (with max panel length constraints)
# ================================
from collections import Counter, defaultdict
from typing import Tuple, List, Dict
import math

# ---- USER-CONFIGURABLE LIMITS (in meters) ----
# Set these according to factory & transport conditions.
# If you don't want a specific limit, set it to None.
MAX_PANEL_LENGTH_FACTORY_M   = 22.45   # example: max 12 m from factory
MAX_PANEL_LENGTH_TRANSPORT_M = 12.5   # example: max 13.5 m for transport
# Effective max length = min(factory, transport) for each orientation (in meters).

# ---- numeric helpers ----
def _round6(x: float) -> float:
    return round(float(x), 6)

def _pt_norm(p: Tuple[float,float]) -> Tuple[float,float]:
    return (_round6(p[0]), _round6(p[1]))

def _edge_norm(a: Tuple[float,float], b: Tuple[float,float]) -> Tuple[Tuple[float,float],Tuple[float,float]]:
    """Undirected, endpoint-sorted edge key with 6-dec rounding."""
    a = _pt_norm(a); b = _pt_norm(b)
    return (a, b) if a <= b else (b, a)

def _edge_oriented(a, b):
    """Return edge ordered left→right for H, bottom→top for V (assumes axis-aligned)."""
    a = _pt_norm(a); b = _pt_norm(b)
    if _round6(a[1]) == _round6(b[1]):           # horizontal
        return (a, b) if a[0] <= b[0] else (b, a)
    else:                                         # vertical
        return (a, b) if a[1] <= b[1] else (b, a)

# ---- geometry for a single cell ----
def _cell_box_from_node(G_cells, cell_id):
    """Return (x0,y0,x1,y1) for the axis-aligned rectangle of a cell id."""
    cx, cy = G_cells.nodes[cell_id]['coord']
    w, h   = G_cells.nodes[cell_id]['size']
    x0, y0 = cx - w/2.0, cy - h/2.0
    x1, y1 = cx + w/2.0, cy + h/2.0
    return (_round6(x0), _round6(y0), _round6(x1), _round6(y1)), (w, h)

def _unit_edges_of_cell(G_cells, cell_id):
    """Return list of the 4 unit edges as ((x,y),(x,y)) with normalized rounding."""
    (x0,y0,x1,y1), _ = _cell_box_from_node(G_cells, cell_id)
    top    = ((_round6(x0), _round6(y1)), (_round6(x1), _round6(y1)))
    bottom = ((_round6(x0), _round6(y0)), (_round6(x1), _round6(y0)))
    left   = ((_round6(x0), _round6(y0)), (_round6(x0), _round6(y1)))
    right  = ((_round6(x1), _round6(y0)), (_round6(x1), _round6(y1)))
    return [top, bottom, left, right]

# ---- perimeter extraction for one room ----
def _room_perimeter_unit_edges(G_cells, room_cells: List[str]) -> List[Tuple[Tuple[float,float],Tuple[float,float]]]:
    """
    Classic trick: collect all unit edges of all cells; perimeter edges are those with count==1.
    """
    edge_counts = Counter()
    for cid in room_cells:
        for e in _unit_edges_of_cell(G_cells, cid):
            edge_counts[_edge_norm(*e)] += 1
    # perimeter = appear once
    return [e for e, cnt in edge_counts.items() if cnt == 1]

# ---- group collinear unit edges into maximal panels (respect max lengths) ----
def _group_panels_from_unit_edges(
    unit_edges: List[Tuple[Tuple[float,float],Tuple[float,float]]],
    cell_size_xy: Tuple[float,float],
    max_len_factory: float | None = None,
    max_len_transport: float | None = None
):
    """
    Input:
        unit_edges: list of undirected unit edges (normalized endpoints).
        cell_size_xy: (w, h) grid cell size in meters.
        max_len_factory: maximum panel length allowed by factory (m) or None
        max_len_transport: maximum panel length allowed by transport (m) or None

    Output:
        list of panels: dicts with:
            - orientation ('H' or 'V')
            - start, end
            - edges (oriented sub-chain)
            - length_cells, length_m
    """
    if not unit_edges:
        return []

    w, h = cell_size_xy

    # Determine effective max length (m) and convert to max number of cells per orientation.
    def _min_positive(vals):
        vals = [v for v in vals if v is not None]
        return min(vals) if vals else None

    max_len_m = _min_positive([max_len_factory, max_len_transport])
    if max_len_m is None:
        max_H_cells = None
        max_V_cells = None
    else:
        # At least 1 cell if w/h > 0
        max_H_cells = max(1, int(math.floor(max_len_m / w))) if w > 0 else None
        max_V_cells = max(1, int(math.floor(max_len_m / h))) if h > 0 else None

    # Split edges by orientation and constant coordinate
    H_lines = defaultdict(list)   # key=y_value -> list of oriented edges (left->right)
    V_lines = defaultdict(list)   # key=x_value -> list of oriented edges (bottom->top)

    for e in unit_edges:
        (a, b) = e
        if _round6(a[1]) == _round6(b[1]):   # horizontal
            oa, ob = _edge_oriented(a, b)
            y = oa[1]
            H_lines[y].append((oa, ob))
        else:                                # vertical
            oa, ob = _edge_oriented(a, b)
            x = oa[0]
            V_lines[x].append((oa, ob))

    panels = []

    def merge_line(edges_oriented, axis='H'):
        """
        Edges_oriented are consecutive unit edges along one straight line;
        we first merge fully contiguous runs, then later split by max length.
        """
        # sort by the progressing coordinate (x for H, y for V)
        if axis == 'H':
            edges_oriented.sort(key=lambda e: e[0][0])  # sort by left x
        else:
            edges_oriented.sort(key=lambda e: e[0][1])  # sort by bottom y

        chain = []
        for (a, b) in edges_oriented:
            if not chain:
                chain = [(a, b)]
            else:
                prev_a, prev_b = chain[-1]
                if prev_b == a:         # perfectly adjacent
                    chain.append((a, b))
                else:
                    # yield the chain built so far, then start a new one
                    yield chain
                    chain = [(a, b)]
        if chain:
            yield chain

    # Horizontal lines → panels (split by max_H_cells if needed)
    for y, edges in H_lines.items():
        for chain in merge_line(edges, axis='H'):
            n = len(chain)
            if (max_H_cells is None) or (n <= max_H_cells):
                subchains = [chain]
            else:
                # Split long chain into chunks of max_H_cells
                subchains = [chain[i:i+max_H_cells] for i in range(0, n, max_H_cells)]

            for sub in subchains:
                start = sub[0][0]
                end   = sub[-1][1]
                length_cells = len(sub)
                panels.append({
                    "orientation": "H",
                    "start": start,
                    "end": end,
                    "edges": sub,
                    "length_cells": length_cells,
                    "length_m": length_cells * w
                })

    # Vertical lines → panels (split by max_V_cells if needed)
    for x, edges in V_lines.items():
        for chain in merge_line(edges, axis='V'):
            n = len(chain)
            if (max_V_cells is None) or (n <= max_V_cells):
                subchains = [chain]
            else:
                # Split long chain into chunks of max_V_cells
                subchains = [chain[i:i+max_V_cells] for i in range(0, n, max_V_cells)]

            for sub in subchains:
                start = sub[0][0]
                end   = sub[-1][1]
                length_cells = len(sub)
                panels.append({
                    "orientation": "V",
                    "start": start,
                    "end": end,
                    "edges": sub,
                    "length_cells": length_cells,
                    "length_m": length_cells * h
                })

    return panels

# ---- matelines from panel endpoints ----
def _compute_matelines(panels: List[Dict]) -> List[Dict]:
    """
    A mateline is a point where >= 2 different panels meet (by endpoint coincidence).
    """
    pt_to_panels = defaultdict(set)
    for p in panels:
        s = _pt_norm(p["start"])
        e = _pt_norm(p["end"])
        pt_to_panels[s].add(p["panel_id"])
        pt_to_panels[e].add(p["panel_id"])

    matelines = []
    for pt, ids in pt_to_panels.items():
        if len(ids) >= 2:
            matelines.append({
                "point": pt,
                "incident_panels": sorted(ids),
                "degree": len(ids)
            })
    return matelines

# ---- main: panelize ONE layout_dict (UID -> cells) ----
def panelize_layout(
    layout_dict: Dict[str, List[str]],
    G_cells,
    id_prefix: str = "P",
    max_len_factory: float | None = MAX_PANEL_LENGTH_FACTORY_M,
    max_len_transport: float | None = MAX_PANEL_LENGTH_TRANSPORT_M
) -> Dict:
    """
    Returns PanelizationResult:
      {
        "panels": [Panel,...],
        "matelines": [Mateline,...],
        "summary": {...}
      }

    Panel lengths are capped by max_len_factory / max_len_transport:
      effective_max_length = min(factory, transport) in meters.
      Long continuous chains are split into smaller panels that respect this limit.
    """
    # choose a representative cell to read grid step (assume uniform)
    any_node = next(iter(G_cells.nodes))
    _, (w, h) = _cell_box_from_node(G_cells, any_node)

    all_panels = []
    panels_by_room = defaultdict(list)
    panel_counter = 0

    for room_uid, room_cells in layout_dict.items():
        # (A) perimeter unit edges for this room
        unit_edges = _room_perimeter_unit_edges(G_cells, room_cells)

        # (B) group into maximal straight panels with length constraints
        room_panels = _group_panels_from_unit_edges(
            unit_edges,
            (w, h),
            max_len_factory=max_len_factory,
            max_len_transport=max_len_transport
        )

        # (C) assign IDs and annotate with room info
        for rp in room_panels:
            panel_counter += 1
            rp["panel_id"] = f"{id_prefix}{panel_counter}"
            rp["room_uid"] = room_uid
            all_panels.append(rp)
            panels_by_room[room_uid].append(rp["panel_id"])

    # (D) matelines from endpoints
    matelines = _compute_matelines(all_panels)

    # (E) summary
    length_hist = Counter(p["length_cells"] for p in all_panels)
    result = {
        "panels": all_panels,
        "matelines": matelines,
        "summary": {
            "total_panels": len(all_panels),
            "total_matelines": len(matelines),
            "panel_length_hist_cells": dict(sorted(length_hist.items())),
            "panels_by_room": dict(panels_by_room)
        }
    }
    return result

# ---- batch: panelize EVERY layout in valid_layouts_step8 ----
def panelize_all(
    valid_layouts_step8,
    G_cells,
    max_len_factory: float | None = MAX_PANEL_LENGTH_FACTORY_M,
    max_len_transport: float | None = MAX_PANEL_LENGTH_TRANSPORT_M
):
    """
    Input:
      valid_layouts_step8: list of (layout_dict, logic, relationships, incomplete)

    Output:
      panelized_layouts: list of (layout_dict, logic, relationships, incomplete, panelization_result)
      panel_results_only: list of panelization_result (same order)
    """
    out = []
    res_only = []
    for (layout_dict, logic, relationships, incomplete) in valid_layouts_step8:
        pz = panelize_layout(
            layout_dict,
            G_cells,
            id_prefix="P",
            max_len_factory=max_len_factory,
            max_len_transport=max_len_transport
        )
        out.append((layout_dict, logic, relationships, incomplete, pz))
        res_only.append(pz)
    return out, res_only

# ================================
# Example: run it on your Step 8 outputs
# ================================
panelized_layouts, panel_results = panelize_all(
    valid_layouts_step8,
    G_cells,
    max_len_factory=MAX_PANEL_LENGTH_FACTORY_M,
    max_len_transport=MAX_PANEL_LENGTH_TRANSPORT_M
)

# Quick sanity prints for the first few
for i, pz in enumerate(panel_results[:3]):
    sm = pz["summary"]
    print(f"[Layout {i}] panels={sm['total_panels']} | matelines={sm['total_matelines']} | length_hist={sm['panel_length_hist_cells']}")
    # If you want to verify visually which points count as matelines:
    # print("  Mateline points:", [m['point'] for m in pz['matelines']][:10])

# =========================================
# Panel + Mateline Browser for your layouts
# =========================================
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def _draw_rooms(ax, layout_dict, G_cells, plot_polygon=None, building_polygon=None,
                annotate_nodes=False, room_alpha=0.60):

    if building_polygon is not None:
        ax.add_patch(
            patches.Polygon(np.array(building_polygon.exterior.coords),
                            fill=True, color='lightgrey', alpha=0.30)
        )
    if plot_polygon is not None:
        ax.add_patch(
            patches.Polygon(np.array(plot_polygon.exterior.coords),
                            fill=False, edgecolor='gray', linestyle='--', linewidth=0.9)
        )

    cmap = plt.cm.get_cmap('tab20', max(1, len(layout_dict)))
    for i, (room_uid, nodes) in enumerate(layout_dict.items()):
        color = cmap(i)
        for n in nodes:
            cx, cy = G_cells.nodes[n]['coord']
            w, h   = G_cells.nodes[n]['size']
            ax.add_patch(
                patches.Rectangle((cx - w/2, cy - h/2), w, h,
                                  facecolor=color, edgecolor='black',
                                  linewidth=0.25, alpha=room_alpha)
            )
            if annotate_nodes:
                ax.text(cx, cy, str(n), fontsize=4.5, ha='center', va='center')


def _midpoint(a, b):
    return ((a[0] + b[0]) / 2.0, (a[1] + b[1]) / 2.0)

def _draw_panels_and_matelines(ax, panelization, show_panel_ids=False,
                               panel_linewidth=2.0, mateline_ms=7):

    panels = panelization["panels"]
    matelines = panelization["matelines"]

    # Color panels by length_cells (same-length panels share color)
    lengths = sorted({p["length_cells"] for p in panels})
    length_to_color = {L: plt.cm.tab10(i % 10) for i, L in enumerate(lengths)}

    # Panels
    for p in panels:
        s, e = p["start"], p["end"]
        ax.plot([s[0], e[0]], [s[1], e[1]],
                linewidth=panel_linewidth,
                color=length_to_color[p["length_cells"]],
                solid_capstyle='round',
                zorder=6)
        if show_panel_ids:
            mx, my = _midpoint(s, e)
            ax.text(mx, my, p["panel_id"], fontsize=6, color='black',
                    ha='center', va='center',
                    bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.75),
                    zorder=7)

    # Compact legend: show up to 6 length categories only
    if lengths:
        shown = lengths[:6]
        handles = [patches.Patch(color=length_to_color[L], label=f"len={L} cells") for L in shown]
        if len(lengths) > 6:
            handles.append(patches.Patch(fc='none', ec='none', label=f"+{len(lengths)-6} more"))
        ax.legend(handles=handles, loc='upper right', fontsize=7, frameon=False)

    # Matelines
    if matelines:
        xs = [m["point"][0] for m in matelines]
        ys = [m["point"][1] for m in matelines]
        ax.scatter(xs, ys, marker='x',
                   s=(mateline_ms**2),
                   linewidths=1.6,
                   color='magenta',
                   zorder=8)


def visualize_panelized_grid(panelized_layouts,
                             G_cells, building_polygon, plot_polygon,
                             cols=3, rows=2, start=0,
                             show_panel_ids=False, annotate_nodes=False,
                             export_page=True):
    per_page = cols * rows
    end = min(len(panelized_layouts), start + per_page)
    page = panelized_layouts[start:end]
    if not page:
        print("No layouts to display in this range.")
        return

    fig, axes = plt.subplots(rows, cols, figsize=(4.7*cols, 3.9*rows))
    if rows*cols == 1:
        axes = np.array([[axes]])
    elif rows == 1 or cols == 1:
        axes = np.array(axes).reshape(rows, cols)

    xmin, ymin, xmax, ymax = plot_polygon.bounds

    for k, ax in enumerate(axes.flat):
        ax.clear()
        if k >= len(page):
            ax.axis('off')
            continue

        layout_dict, logic, relationships, incomplete, pz = page[k]

        _draw_rooms(ax, layout_dict, G_cells,
                    plot_polygon=plot_polygon, building_polygon=building_polygon,
                    annotate_nodes=annotate_nodes, room_alpha=0.60)

        _draw_panels_and_matelines(ax, pz,
                                   show_panel_ids=show_panel_ids,
                                   panel_linewidth=2.0,
                                   mateline_ms=7)

        sm = pz["summary"]
        title = f"L={logic} | {'INC' if incomplete else 'OK'} | panels={sm['total_panels']} | matelines={sm['total_matelines']}"
        ax.set_title(title, fontsize=9)
        ax.set_aspect('equal', adjustable='box')
        ax.set_xlim(xmin - 0.8, xmax + 0.8)
        ax.set_ylim(ymin - 0.8, ymax + 0.8)

        ax.grid(True, linewidth=0.25, alpha=0.3)
        ax.tick_params(labelsize=6)
        ax.set_xlabel("X", fontsize=7)
        ax.set_ylabel("Y", fontsize=7)

    fig.tight_layout()

    if export_page:
        page_tag = f"Step9_Panel_Page_start{start:06d}_cols{cols}_rows{rows}"
        save_figure(fig, page_tag)

    plt.show()

def export_single_panelized_layout(panelized_item,
                                  G_cells, building_polygon, plot_polygon,
                                  step_tag=None,
                                  show_panel_ids=False,
                                  annotate_nodes=False):
    layout_dict, logic, relationships, incomplete, pz = panelized_item
    sm = pz["summary"]

    if step_tag is None:
        step_tag = f"Step9_Panel_Single_L{logic}_{'INC' if incomplete else 'OK'}_pan{sm['total_panels']}_ml{sm['total_matelines']}"

    fig, ax = plt.subplots(figsize=(8, 8))

    _draw_rooms(ax, layout_dict, G_cells,
                plot_polygon=plot_polygon, building_polygon=building_polygon,
                annotate_nodes=annotate_nodes, room_alpha=0.60)

    _draw_panels_and_matelines(ax, pz,
                               show_panel_ids=show_panel_ids,
                               panel_linewidth=2.2,
                               mateline_ms=8)

    ax.set_title(f"Panelization | L={logic} | panels={sm['total_panels']} | matelines={sm['total_matelines']}",
                 fontsize=10)

    xmin, ymin, xmax, ymax = plot_polygon.bounds
    ax.set_xlim(xmin - 0.8, xmax + 0.8)
    ax.set_ylim(ymin - 0.8, ymax + 0.8)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, linewidth=0.25, alpha=0.3)
    ax.tick_params(labelsize=7)
    ax.set_xlabel("X", fontsize=8)
    ax.set_ylabel("Y", fontsize=8)

    save_figure(fig, step_tag)
    plt.show()


def browse_panelized_layouts(panelized_layouts,
                             G_cells, building_polygon, plot_polygon,
                             cols=3, rows=2, show_panel_ids=True, annotate_nodes=False):
    """
    Simple pager: [Enter] next | p prev | q quit | # jump
    """
    per_page = cols * rows
    start = 0
    n = len(panelized_layouts)
    if n == 0:
        print("Nothing to browse: panelized_layouts is empty.")
        return
    while True:
        print(f"\nShowing panelized layouts {start}..{min(n-1, start+per_page-1)} of {n-1}")
        visualize_panelized_grid(panelized_layouts, G_cells, building_polygon, plot_polygon,
                                 cols=cols, rows=rows, start=start,
                                 show_panel_ids=show_panel_ids, annotate_nodes=annotate_nodes)
        cmd = input("[Enter] next | p prev | q quit | #: jump start index → ").strip().lower()
        if cmd == "q":
            break
        elif cmd == "p":
            start = max(0, start - per_page)
        elif cmd == "":
            start = min(max(0, n-1), start + per_page)
        else:
            try:
                j = int(cmd)
                if 0 <= j < n:
                    start = (j // per_page) * per_page
                else:
                    print("Index out of range.")
            except:
                print("Unrecognized command.")

# You already have:
# panelized_layouts (from panelize_all)
# G_cells, selected_building, spatial_data['polygon']

browse_panelized_layouts(
    panelized_layouts,
    G_cells,
    selected_building,
    spatial_data['polygon'],
    cols=3, rows=2,
    show_panel_ids=True,     # turn off if you want cleaner plots
    annotate_nodes=False     # set True to see cell IDs inside rooms
)

export_single_panelized_layout(panelized_layouts[0], G_cells, selected_building, spatial_data['polygon'],
                              show_panel_ids=False)
export_single_panelized_layout(panelized_layouts[1], G_cells, selected_building, spatial_data['polygon'],
                              show_panel_ids=False)
export_single_panelized_layout(panelized_layouts[2], G_cells, selected_building, spatial_data['polygon'],
                              show_panel_ids=False)

***Step 10: GA Optimization***

***10.0: Benchmark Controls and Libraries***

In [45]:
import copy, math, random, time
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D

from shapely.geometry import Point, Polygon
from shapely.ops import unary_union

# ==========================
# BENCHMARK CONTROL (same for GA/PSO/ACO/SA)
# ==========================
BENCH_SEED   = 42
POP_SIZE     = 40
EVAL_BUDGET  = 2000  # same across algorithms

# Score directions
MAX_KEYS = ("area_public", "area_private")
MIN_KEYS = (
    "circ_cost", "zoning_pen", "compactness", "egress_pen",
    "panel_count_total", "panel_unique_types", "matelines_total"
)

def set_benchmark_seed(seed=BENCH_SEED):
    random.seed(seed)
    np.random.seed(seed)

def banner(title: str):
    line = "=" * 78
    print("\n" + line)
    print(title)
    print(line)

***10.1: Utilities, Objectives and Panelization Metrics***

In [46]:
# -----------------------------
# Utilities
# -----------------------------
def base_of(name_with_uid: str) -> str:
    return name_with_uid.split('#', 1)[0].strip()

def _cell_poly(G_cells, node):
    cx, cy = G_cells.nodes[node]['coord']
    w, h   = G_cells.nodes[node]['size']
    return Polygon([
        (cx - w/2, cy - h/2),
        (cx + w/2, cy - h/2),
        (cx + w/2, cy + h/2),
        (cx - w/2, cy + h/2),
    ])

def get_room_polygon(G_cells, nodes: List[str]):
    if not nodes:
        return Polygon()
    return unary_union([_cell_poly(G_cells, n) for n in nodes])

def _cell_area(G_cells) -> float:
    n0 = next(iter(G_cells.nodes))
    w, h = G_cells.nodes[n0]['size']
    return float(w) * float(h)

# -----------------------------
# Objective features
# -----------------------------
PUBLIC_TAGS  = ("Living", "Dining", "Kitchen")
PRIVATE_TAGS = ("Bathroom", "Master Bedroom (No Cabinet)", "Second Bedroom (No Cabinet)", "Bedroom")

def features_for_objectives(
    G_cells,
    layout: Dict[str, List[str]],
    building_polygon: Polygon,
    entrance_node: str | None = None,
    public_tags=PUBLIC_TAGS,
    private_tags=PRIVATE_TAGS
) -> Dict[str, float]:

    polys = {uid: get_room_polygon(G_cells, ns) for uid, ns in layout.items()}

    # area by zone
    area_public = 0.0
    area_private = 0.0
    for uid, poly in polys.items():
        b = base_of(uid).lower()
        if any(tag.lower() in b for tag in public_tags):
            area_public += float(poly.area)
        if any(tag.lower() in b for tag in private_tags):
            area_private += float(poly.area)

    # compactness = unused fraction inside building polygon
    B = float(building_polygon.area)
    assigned = float(sum(p.area for p in polys.values()))
    compactness = max(0.0, (B - assigned) / max(1e-9, B))

    # egress penalty: rooms without exterior boundary exposure
    bnd = building_polygon.boundary
    egress_pen = 0.0
    for poly in polys.values():
        if poly.is_empty or poly.boundary.intersection(bnd).length <= 1e-9:
            egress_pen += 1.0

    # circulation cost: sum distance from entrance (or building centroid)
    if entrance_node and entrance_node in G_cells.nodes:
        ex, ey = G_cells.nodes[entrance_node]['coord']
        ent = Point(ex, ey)
    else:
        ent = building_polygon.centroid

    circ_cost = float(sum(ent.distance(poly.centroid) for poly in polys.values() if not poly.is_empty))

    # zoning penalty placeholder
    zoning_pen = 0.0

    return {
        "area_public": area_public,
        "area_private": area_private,
        "compactness": compactness,
        "egress_pen": egress_pen,
        "circ_cost": circ_cost,
        "zoning_pen": zoning_pen,
    }

# -----------------------------
# Panelization metrics (uses panelize_layout if available; fallback otherwise)
# -----------------------------
def _collect_wall_segments(G_cells, layout_uid_dict):
    node2room = {}
    for uid, nodes in layout_uid_dict.items():
        for n in nodes:
            node2room[n] = uid

    center2node = {tuple(G_cells.nodes[n]['coord']): n for n in G_cells}
    w, h = G_cells.nodes[next(iter(G_cells.nodes))]['size']
    OFF = {"L": (-w, 0), "R": (w, 0), "B": (0, -h), "T": (0, h)}

    def ek(p0, p1):
        p0 = (float(p0[0]), float(p0[1]))
        p1 = (float(p1[0]), float(p1[1]))
        return tuple(sorted((p0, p1)))

    segs = set()
    for n, uid in node2room.items():
        cx, cy = G_cells.nodes[n]['coord']
        x0, y0, x1, y1 = cx - w/2, cy - h/2, cx + w/2, cy + h/2

        faces = {
            "L": ((x0, y0), (x0, y1)),
            "R": ((x1, y0), (x1, y1)),
            "B": ((x0, y0), (x1, y0)),
            "T": ((x0, y1), (x1, y1)),
        }

        for k, (pA, pB) in faces.items():
            dx, dy = OFF[k]
            nbr_center = (round(cx + dx, 6), round(cy + dy, 6))
            nbr = center2node.get(nbr_center)
            if (nbr is None) or (node2room.get(nbr) != uid):
                segs.add(ek(pA, pB))

    return [tuple(s) for s in segs]

def _merge_collinear_segments(segments, tol=1e-6):
    horiz, vert = {}, {}
    for (p0, p1) in segments:
        x0, y0 = p0; x1, y1 = p1
        if abs(y0 - y1) < tol:
            y = round(y0, 6)
            a, b = sorted([x0, x1])
            horiz.setdefault(y, []).append((a, b))
        elif abs(x0 - x1) < tol:
            x = round(x0, 6)
            a, b = sorted([y0, y1])
            vert.setdefault(x, []).append((a, b))

    def merge(intervals):
        if not intervals:
            return []
        intervals.sort()
        out = [list(intervals[0])]
        for a, b in intervals[1:]:
            if a <= out[-1][1] + 1e-9:
                out[-1][1] = max(out[-1][1], b)
            else:
                out.append([a, b])
        return [tuple(x) for x in out]

    runs = []
    for y, xs in horiz.items():
        for a, b in merge(xs):
            runs.append(((a, y), (b, y)))
    for x, ys in vert.items():
        for a, b in merge(ys):
            runs.append(((x, a), (x, b)))
    return runs

def _fallback_panelization(layout, G_cells):
    runs = _merge_collinear_segments(_collect_wall_segments(G_cells, layout))
    endpoints = set()
    for p0, p1 in runs:
        endpoints.add((round(p0[0], 6), round(p0[1], 6)))
        endpoints.add((round(p1[0], 6), round(p1[1], 6)))

    w, h = G_cells.nodes[next(iter(G_cells.nodes))]['size']
    module = min(float(w), float(h))
    lengths_cells = []
    for (a, b) in runs:
        Lm = math.hypot(b[0] - a[0], b[1] - a[1])
        lengths_cells.append(int(round(Lm / module)))

    return {"total_panels": len(runs), "total_matelines": len(endpoints), "_lengths_cells": lengths_cells}

def panel_metrics_from_layout(layout: Dict[str, List[str]], G_cells):
    try:
        pz = panelize_layout(layout, G_cells)  # from Step 9
        sm = pz["summary"]
        unique_types = len(set(int(p.get("length_cells", 0)) for p in pz.get("panels", [])))
        return {
            "panel_count_total": float(sm.get("total_panels", 0.0)),
            "panel_unique_types": float(unique_types),
            "matelines_total": float(sm.get("total_matelines", 0.0)),
            "panelization": pz,
        }
    except Exception:
        sm = _fallback_panelization(layout, G_cells)
        unique_types = len(set(sm.get("_lengths_cells", [])))
        return {
            "panel_count_total": float(sm["total_panels"]),
            "panel_unique_types": float(unique_types),
            "matelines_total": float(sm["total_matelines"]),
            "panelization": {"summary": sm, "panels": [], "matelines": []},
        }

# -----------------------------
# Scoring weights & scalar score
# -----------------------------
def make_weights(
    W_PUBLIC=1.0, W_PRIVATE=1.0,
    W_CIRC=1.0, W_ZONING=1.0, W_EGRESS=1.0,
    W_COMPACT=1.0,
    W_PANELS=1.0, W_TYPES=1.0, W_MATES=1.0
):
    return {
        "area_public": float(W_PUBLIC),
        "area_private": float(W_PRIVATE),
        "circ_cost": float(W_CIRC),
        "zoning_pen": float(W_ZONING),
        "egress_pen": float(W_EGRESS),
        "compactness": float(W_COMPACT),
        "panel_count_total": float(W_PANELS),
        "panel_unique_types": float(W_TYPES),
        "matelines_total": float(W_MATES),
    }

GA_WEIGHTS = make_weights()

def score_from_components(comps_raw: dict, weights: dict) -> float:
    s = 0.0
    for k in MAX_KEYS:
        s += float(weights.get(k, 0.0)) * float(comps_raw.get(k, 0.0))
    for k in MIN_KEYS:
        s -= float(weights.get(k, 0.0)) * float(comps_raw.get(k, 0.0))
    return float(s)

***10.2: Deterministic Seeding, Utilities and Layout Drawing***

In [49]:
# -----------------------------
# Deterministic seeding from panelized_layouts
# -----------------------------
def seed_from_panelized(panelized_layouts, pop_size=POP_SIZE):
    if not panelized_layouts:
        return []

    def fp(L):
        return tuple((k, tuple(sorted(v))) for k, v in sorted(L.items()))

    uniq = {}
    for tpl in panelized_layouts:
        try:
            L, lg = tpl[0], int(tpl[1])
        except Exception:
            continue
        key = fp(L)
        if key not in uniq:
            uniq[key] = {"layout": copy.deepcopy(L), "logic": lg}

    keys_sorted = sorted(uniq.keys())
    seeds = [uniq[k] for k in keys_sorted]
    if not seeds:
        return []

    out = []
    i = 0
    while len(out) < pop_size:
        out.append(copy.deepcopy(seeds[i % len(seeds)]))
        i += 1
    return out[:pop_size]

# -----------------------------
# Pareto utilities
# -----------------------------
def _vector_for_keys(ind, max_keys: Tuple[str, ...], min_keys: Tuple[str, ...]):
    v = {}
    for k in max_keys:
        v[k] = float(ind["comps_raw"].get(k, 0.0))
    for k in min_keys:
        v[k] = -float(ind["comps_raw"].get(k, 0.0))  # convert to maximize
    return v

def _dominates_keys(a, b, max_keys: Tuple[str, ...], min_keys: Tuple[str, ...], eps=1e-9):
    A = _vector_for_keys(a, max_keys, min_keys)
    B = _vector_for_keys(b, max_keys, min_keys)
    ge = all(A[k] >= B[k] - eps for k in A)
    gt = any(A[k] > B[k] + eps for k in A)
    return ge and gt

def pareto_front_keys(individuals: List[Dict], max_keys: Tuple[str, ...], min_keys: Tuple[str, ...]):
    return [
        a for i, a in enumerate(individuals)
        if not any(_dominates_keys(b, a, max_keys, min_keys) for j, b in enumerate(individuals) if j != i)
    ]

PARETO_MAX_ARCH = ("area_public", "area_private")
PARETO_MIN_ARCH = ("circ_cost", "zoning_pen", "compactness", "egress_pen")

PARETO_MAX_PANEL = tuple()
PARETO_MIN_PANEL = ("panel_count_total", "panel_unique_types", "matelines_total")

# -----------------------------
# Visualization helpers
# -----------------------------
def _draw_layout(ax, layout_dict, G_cells, building_polygon, plot_polygon, title=None):
    cmap = plt.cm.get_cmap('tab20', max(1, len(layout_dict)))

    ax.add_patch(
        patches.Polygon(np.array(building_polygon.exterior.coords),
                        fc='lightgrey', ec='none', alpha=0.85, antialiased=True)
    )
    ax.add_patch(
        patches.Polygon(np.array(plot_polygon.exterior.coords),
                        fill=False, ec='gray', ls='--', lw=1.2, antialiased=True)
    )

    for i, (uid, nodes) in enumerate(layout_dict.items()):
        col = cmap(i)
        for n in nodes:
            cx, cy = G_cells.nodes[n]['coord']
            w, h = G_cells.nodes[n]['size']
            ax.add_patch(
                patches.Rectangle((cx - w/2, cy - h/2), w, h,
                                  fc=col, ec='black', lw=0.35, alpha=0.65, antialiased=True)
            )

    xmin, ymin, xmax, ymax = plot_polygon.bounds
    pad = 0.6
    ax.set_xlim(xmin - pad, xmax + pad)
    ax.set_ylim(ymin - pad, ymax + pad)
    ax.set_aspect('equal')
    ax.grid(True, lw=0.3, alpha=0.35)
    ax.tick_params(labelsize=8)
    if title:
        ax.set_title(title, fontsize=10)

def browse_layout_list(
    layout_list,
    G_cells, building_polygon, plot_polygon,
    cols=3, rows=2, start=0,
    save_pages=False,
    base_name="step10_ga_browse",
    dpi_save=300
):
    per = cols * rows
    end = min(len(layout_list), start + per)
    page = layout_list[start:end]

    fig, axes = plt.subplots(rows, cols, figsize=(4.6 * cols, 3.8 * rows), dpi=300, constrained_layout=True)
    if rows * cols == 1:
        axes = np.array([[axes]])
    elif rows == 1 or cols == 1:
        axes = np.array(axes).reshape(rows, cols)

    for k, ax in enumerate(axes.flat):
        ax.clear()
        if k >= len(page):
            ax.axis('off')
            continue

        ind = page[k]
        c = ind.get("comps_raw", {})
        title = (
            f"score={ind.get('score', 0):.2f} | pub={c.get('area_public', 0):.1f} | "
            f"prv={c.get('area_private', 0):.1f} | "
            f"P/T/M={c.get('panel_count_total', 0):.0f}/"
            f"{c.get('panel_unique_types', 0):.0f}/"
            f"{c.get('matelines_total', 0):.0f}"
        )
        _draw_layout(ax, ind["layout"], G_cells, building_polygon, plot_polygon, title)

    if save_pages:
        page_id = start // per
        save_figure(fig, f"{base_name}_page_{page_id:03d}", dpi=dpi_save)

    plt.show()

***10.3: Initialization***

In [50]:
def GA_Block1_Initialization(panelized_layouts, pop_size=POP_SIZE, seed=BENCH_SEED):
    banner("BLOCK 1 — INITIALIZATION (seed population from Step 9 panelized_layouts)")
    set_benchmark_seed(seed)
    pop = seed_from_panelized(panelized_layouts, pop_size=pop_size)
    print(f"Population size: {len(pop)} (requested {pop_size})")
    if pop:
        logics = [p.get("logic", None) for p in pop]
        print("Logic counts:", dict(pd.Series(logics).value_counts()))
    return pop

***10.4: Initial Evaluation***

In [51]:
def evaluate_population(pop, G_cells, building_polygon, entrance_node, weights):
    for ind in pop:
        f = features_for_objectives(G_cells, ind["layout"], building_polygon, entrance_node)
        p = panel_metrics_from_layout(ind["layout"], G_cells)
        c = {
            **f,
            "panel_count_total": p["panel_count_total"],
            "panel_unique_types": p["panel_unique_types"],
            "matelines_total": p["matelines_total"],
        }
        ind["features"]     = f
        ind["panelization"] = p["panelization"]
        ind["comps_raw"]    = c
        ind["score"]        = score_from_components(c, weights)
    return pop

def export_population_to_excel(pop_sorted, G_cells, selected_building, excel_path):
    # Reuse your robust export style
    def _cell_poly_xy(G_cells, node):
        cx, cy = G_cells.nodes[node]['coord']
        w, h = G_cells.nodes[node]['size']
        return [(cx-w/2, cy-h/2), (cx+w/2, cy-h/2), (cx+w/2, cy+h/2), (cx-w/2, cy+h/2)]

    def _room_area_xy(G_cells, nodes):
        if not nodes:
            return 0.0
        polys = [Polygon(_cell_poly_xy(G_cells, n)) for n in nodes]
        return float(unary_union(polys).area)

    def _panel_types_from_panelization(panelization_dict):
        try:
            sm = panelization_dict.get("summary", {})
            panels = panelization_dict.get("panels", [])
            panel_count_total = float(sm.get("total_panels", 0.0))
            matelines_total = float(sm.get("total_matelines", 0.0))
            types = sorted(set(int(p.get("length_cells", 0)) for p in panels))
            return panel_count_total, float(len(types)), matelines_total, tuple(types)
        except Exception:
            return 0.0, 0.0, 0.0, tuple()

    layout_rows = []
    room_rows = []
    building_area = float(selected_building.area)

    for idx, ind in enumerate(pop_sorted):
        layout = ind["layout"]
        logic  = ind.get("logic", None)
        score  = float(ind.get("score", np.nan))

        comps = ind.get("comps_raw", {})
        panelization = ind.get("panelization", {"summary": {}, "panels": [], "matelines": []})
        panel_count_total, panel_unique_types, matelines_total, types_tuple = _panel_types_from_panelization(panelization)

        assigned_area = sum(_room_area_xy(G_cells, nodes) for nodes in layout.values())
        unused_area = max(0.0, building_area - assigned_area)
        unused_fraction = (unused_area / building_area) if building_area > 0 else np.nan

        layout_rows.append({
            "layout_index": idx,
            "score": score,
            "logic": logic,
            "building_area": building_area,
            "assigned_area": assigned_area,
            "unused_area": unused_area,
            "unused_fraction": unused_fraction,
            "compactness": float(comps.get("compactness", np.nan)),
            "area_public": float(comps.get("area_public", np.nan)),
            "area_private": float(comps.get("area_private", np.nan)),
            "circ_cost": float(comps.get("circ_cost", np.nan)),
            "egress_pen": float(comps.get("egress_pen", np.nan)),
            "zoning_pen": float(comps.get("zoning_pen", np.nan)),
            "panel_count_total": panel_count_total,
            "panel_unique_types": panel_unique_types,
            "matelines_total": matelines_total,
            "panel_types_cells": ",".join(map(str, types_tuple)),
            "num_rooms": len(layout),
        })

        for uid, nodes in layout.items():
            room_rows.append({
                "layout_index": idx,
                "room_uid": uid,
                "room_base": uid.split("#", 1)[0],
                "room_area": _room_area_xy(G_cells, nodes),
                "num_cells": len(nodes),
            })

    df_layouts = pd.DataFrame(layout_rows).sort_values(["score","layout_index"], ascending=[False, True]).reset_index(drop=True)
    df_rooms_long = pd.DataFrame(room_rows)
    df_rooms_wide = (df_rooms_long.pivot_table(index="layout_index", columns="room_uid", values="room_area", aggfunc="sum").reset_index()
                     if not df_rooms_long.empty else pd.DataFrame())

    with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
        df_layouts.to_excel(writer, index=False, sheet_name="layouts_summary")
        df_rooms_long.to_excel(writer, index=False, sheet_name="rooms_long")
        df_rooms_wide.to_excel(writer, index=False, sheet_name="rooms_wide")

    print(f"✅ Exported → {excel_path}  (N={len(df_layouts)})")
    return df_layouts

def GA_Block2_InitialEvaluation(pop, G_cells, building_polygon, selected_building, entrance_node, weights, initial_excel="ga_initial_seed_metrics.xlsx"):
    banner("BLOCK 2 — INITIAL EVALUATION (compute fitness of seed parents + export to Excel)")

    pop = evaluate_population(pop, G_cells, building_polygon, entrance_node, weights)
    pop.sort(key=lambda d: -d["score"])

    best = pop[0]
    c = best["comps_raw"]
    print(f"Best seed score: {best['score']:.3f} | pub={c['area_public']:.1f} prv={c['area_private']:.1f} "
          f"circ={c['circ_cost']:.2f} cmpct={c['compactness']:.2f} eg={c['egress_pen']:.0f} "
          f"P/T/M={c['panel_count_total']:.0f}/{c['panel_unique_types']:.0f}/{c['matelines_total']:.0f}")

    df_initial = export_population_to_excel(pop, G_cells, selected_building, initial_excel)

    return pop, df_initial

***10.5: Selection (Tournament)***

In [52]:
def tournament_select(pop, k=3):
    cand = random.sample(pop, k=min(k, len(pop)))
    cand.sort(key=lambda d: -d["score"])
    return cand[0]

def GA_Block3_Selection(pop, pop_size=POP_SIZE, tournament_k=3, show_selected=True):
    banner("BLOCK 3 — SELECTION (tournament selection)")

    parents = []
    for _ in range(pop_size):
        parents.append(copy.deepcopy(tournament_select(pop, k=tournament_k)))

    parents.sort(key=lambda d: -d["score"])
    print(f"Selected parents: {len(parents)}  | best parent score={parents[0]['score']:.3f}")

    if show_selected:
        print("Showing top selected parents (page 1)…")
        browse_layout_list(parents, G_cells, selected_building, spatial_data['polygon'],
                          cols=3, rows=2, start=0, save_pages=True,
                          base_name="step10_GA_selected_parents", dpi_save=300)
    return parents

***10.6: Crossover (Not coded to keep existing Adjcency and Connectivity Filters)***

In [53]:
def GA_Block4_Crossover(parents, use_crossover=False):
    banner("BLOCK 4 — CROSSOVER (explicit block; OFF in your method)")

    if not use_crossover:
        print("Crossover is DISABLED (consistent with your pipeline). Offspring will be clones of selected parents.")
        offspring = [copy.deepcopy(p) for p in parents]
        return offspring

    # Placeholder (not recommended for layout chromosome unless you design a safe crossover operator)
    print("⚠️ Crossover enabled (placeholder). Using best-of-two cloning to keep feasibility.")
    offspring = []
    for _ in range(len(parents)):
        p1, p2 = random.sample(parents, 2)
        child = copy.deepcopy(p1 if p1["score"] >= p2["score"] else p2)
        offspring.append(child)
    return offspring

***10.7: Mutation (One cell random mutation)***

In [54]:
def _try_place_room(
    G_cells, building_polygon, flood_fill_with_logic,
    layout: Dict[str, List[str]], room_uid: str, base_req: Dict,
    logic: int, target_area: float | None = None, start_nodes=None
):
    req = copy.deepcopy(base_req)
    if target_area is not None:
        req['area'] = float(target_area)

    visited = {n for uid, ns in layout.items() if uid != room_uid for n in ns}
    starts = list(start_nodes or layout.get(room_uid, []))
    random.shuffle(starts)

    for s in starts:
        alloc = flood_fill_with_logic(G_cells, s, req, visited, building_polygon, logic)
        if alloc:
            newL = copy.deepcopy(layout)
            newL[room_uid] = alloc
            return newL
    return None

def mutate_one_cell(
    parent_ind,
    G_cells, building_polygon, room_constraints,
    flood_fill_with_logic,
    mutation_rate=0.6,
    expand_prob=0.4, move_prob=0.4, shrink_prob=0.2,
    flip_prob=0.1
):
    if random.random() > mutation_rate:
        return copy.deepcopy(parent_ind)

    layout = copy.deepcopy(parent_ind["layout"])
    logic  = int(parent_ind["logic"])

    rooms = list(layout.keys())
    if not rooms:
        return {"layout": layout, "logic": logic}

    # optional flip logic
    if random.random() < flip_prob:
        logic = {1: 2, 2: 1, 3: 4, 4: 3}.get(logic, logic)

    room = random.choice(rooms)
    base = base_of(room)
    base_req = room_constraints.get(base)
    if not base_req:
        return {"layout": layout, "logic": logic}

    cellA = _cell_area(G_cells)
    cur_cells = len(layout[room])
    cur_area  = cur_cells * cellA

    r = random.random()
    if r < expand_prob:
        target = cur_area + cellA
        newL = _try_place_room(G_cells, building_polygon, flood_fill_with_logic,
                               layout, room, base_req, logic,
                               target_area=target, start_nodes=layout[room])
        if newL:
            return {"layout": newL, "logic": logic}

    elif r < expand_prob + move_prob:
        target = cur_area
        newL = _try_place_room(G_cells, building_polygon, flood_fill_with_logic,
                               layout, room, base_req, logic,
                               target_area=target, start_nodes=layout[room])
        if newL:
            return {"layout": newL, "logic": logic}

    else:
        target = max(float(base_req['area']), cur_area - cellA)
        newL = _try_place_room(G_cells, building_polygon, flood_fill_with_logic,
                               layout, room, base_req, logic,
                               target_area=target, start_nodes=layout[room])
        if newL:
            return {"layout": newL, "logic": logic}

    return {"layout": layout, "logic": logic}

def GA_Block5_Mutation(offspring, mutation_rate=0.6):
    banner("BLOCK 5 — MUTATION (one-cell expand/move/shrink + flood-fill repair)")

    mutated = []
    for ind in offspring:
        child = mutate_one_cell(ind, G_cells, selected_building, room_constraints,
                                flood_fill_with_logic, mutation_rate=mutation_rate)
        mutated.append(child)

    print(f"Mutated offspring generated: {len(mutated)}")
    return mutated

***10.8: Elitism (Carry the best solutions)***

In [55]:
def GA_Block6_Elitism(current_pop_sorted, next_pop, elite_frac=0.25):
    banner("BLOCK 6 — ELITISM (carry best from previous generation)")

    elite_n = max(1, int(elite_frac * len(current_pop_sorted)))
    elites = [copy.deepcopy(ind) for ind in current_pop_sorted[:elite_n]]

    # Replace first elite_n individuals in next_pop
    next_pop = list(next_pop)
    for i in range(min(elite_n, len(next_pop))):
        next_pop[i] = elites[i]

    print(f"Elites carried: {elite_n}")
    return next_pop

***10.9: Main GA Loop***

In [56]:
def GA_Block7_FindBest(pop):
    best = max(pop, key=lambda d: d["score"])
    c = best["comps_raw"]
    print(f"Best score={best['score']:.3f} | pub={c['area_public']:.1f} prv={c['area_private']:.1f} "
          f"circ={c['circ_cost']:.2f} cmpct={c['compactness']:.2f} eg={c['egress_pen']:.0f} "
          f"P/T/M={c['panel_count_total']:.0f}/{c['panel_unique_types']:.0f}/{c['matelines_total']:.0f}")
    return best

def run_GA_7blocks(
    panelized_layouts,
    G_cells,
    building_polygon,
    selected_building,
    room_constraints,
    flood_fill_with_logic,
    entrance_node=None,
    pop_size=POP_SIZE,
    eval_budget=EVAL_BUDGET,
    seed=BENCH_SEED,
    weights=GA_WEIGHTS,
    elite_frac=0.25,
    tournament_k=3,
    mutation_rate=0.6,
    use_crossover=False,
    verbose=True
):
    t0 = time.perf_counter()
    eval_count = 0
    history = []

    # Block 1
    pop = GA_Block1_Initialization(panelized_layouts, pop_size=pop_size, seed=seed)

    # Block 2 (initial evaluation + export initial Excel)
    pop, df_initial = GA_Block2_InitialEvaluation(pop, G_cells, building_polygon,
                                                  selected_building, entrance_node,
                                                  weights, initial_excel="ga_initial_seed_metrics.xlsx")
    eval_count += len(pop)

    best0 = pop[0]
    history.append({"gen": 0, "eval": eval_count, "best_score": best0["score"], "avg_score": float(np.mean([p["score"] for p in pop]))})

    gen = 0
    while eval_count < eval_budget:
        gen += 1
        if verbose:
            banner(f"GA GENERATION {gen:03d}  (eval={eval_count}/{eval_budget})")

        pop.sort(key=lambda d: -d["score"])  # ensure sorted

        # Block 3: selection
        parents = GA_Block3_Selection(pop, pop_size=pop_size, tournament_k=tournament_k, show_selected=(gen == 1))

        # Block 4: crossover
        offspring = GA_Block4_Crossover(parents, use_crossover=use_crossover)

        # Block 5: mutation
        offspring = GA_Block5_Mutation(offspring, mutation_rate=mutation_rate)

        # Evaluate offspring (use same evaluator)
        offspring = evaluate_population(offspring, G_cells, building_polygon, entrance_node, weights)
        eval_count += len(offspring)

        # Block 6: elitism
        offspring.sort(key=lambda d: -d["score"])
        pop = GA_Block6_Elitism(current_pop_sorted=pop, next_pop=offspring, elite_frac=elite_frac)

        # Re-sort after elitism
        pop.sort(key=lambda d: -d["score"])

        # Block 7: best of generation
        best = GA_Block7_FindBest(pop)
        history.append({"gen": gen, "eval": eval_count, "best_score": best["score"], "avg_score": float(np.mean([p["score"] for p in pop]))})

        if eval_count >= eval_budget:
            break

    # final pareto sets
    pop.sort(key=lambda d: -d["score"])
    pareto_sets = {
        "arch":  pareto_front_keys(pop, PARETO_MAX_ARCH,  PARETO_MIN_ARCH),
        "panel": pareto_front_keys(pop, PARETO_MAX_PANEL, PARETO_MIN_PANEL),
    }

    runtime_sec = float(time.perf_counter() - t0)
    run_info = {"runtime_sec": runtime_sec, "eval_count": int(eval_count), "generations": int(gen)}

    return pop, pareto_sets, run_info, pd.DataFrame(history), df_initial

***10.10: Run GA***

In [ ]:
ga_sorted, pareto_sets, run_info, ga_history_df, df_initial = run_GA_7blocks(
    panelized_layouts=panelized_layouts,
    G_cells=G_cells,
    building_polygon=selected_building,
    selected_building=selected_building,
    room_constraints=room_constraints,
    flood_fill_with_logic=flood_fill_with_logic,
    entrance_node=None,
    pop_size=POP_SIZE,
    eval_budget=EVAL_BUDGET,
    seed=BENCH_SEED,
    weights=GA_WEIGHTS,
    elite_frac=0.25,
    tournament_k=3,
    mutation_rate=0.6,
    use_crossover=False,
    verbose=True
)

print(f"\n✅ GA complete | best_score={ga_sorted[0]['score']:.3f} | "
      f"evals={run_info['eval_count']} | time={run_info['runtime_sec']:.2f}s | gens={run_info['generations']}")

***10.11: Export Final GA Population to Excel***

In [61]:
df_final = export_population_to_excel(ga_sorted, G_cells, selected_building, "ga_layout_metrics.xlsx")

✅ Exported → ga_layout_metrics.xlsx  (N=40)


***10.12: Initial vs Final Result***

In [ ]:
banner("INITIAL vs FINAL COMPARISON (score distribution)")

df_init = pd.read_excel("ga_initial_seed_metrics.xlsx", sheet_name="layouts_summary")
df_fin  = pd.read_excel("ga_layout_metrics.xlsx",       sheet_name="layouts_summary")

init_scores = np.sort(df_init["score"].values)[::-1]
fin_scores  = np.sort(df_fin["score"].values)[::-1]

fig, ax = plt.subplots(figsize=(7,4), dpi=300, constrained_layout=True)
ax.plot(np.arange(1, len(init_scores)+1), init_scores, lw=1.5, label="Initial seeds (Step 9)")
ax.plot(np.arange(1, len(fin_scores)+1),  fin_scores,  lw=1.5, label="After GA (Step 10)")
ax.set_xlabel("Rank (best to worst)")
ax.set_ylabel("Score")
ax.set_title("Score Curve — Initial vs After GA")
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
ax.legend(fontsize=9)
save_figure(fig, "ga_initial_vs_final_scorecurve", dpi=600)
plt.show()

fig, ax = plt.subplots(figsize=(7,4), dpi=300, constrained_layout=True)
ax.hist(df_init["score"].values, bins=10, alpha=0.6, edgecolor="black", label="Initial seeds")
ax.hist(df_fin["score"].values,  bins=10, alpha=0.6, edgecolor="black", label="After GA")
ax.set_xlabel("Score")
ax.set_ylabel("Count")
ax.set_title("Score Histogram — Initial vs After GA")
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
ax.legend(fontsize=9)
save_figure(fig, "ga_initial_vs_final_hist", dpi=600)
plt.show()

***10.13: Parallel Coordinates***

In [ ]:
excel_path = "ga_layout_metrics.xlsx"
sheet_name = "layouts_summary"

cols = ["score","area_public","area_private","circ_cost","panel_count_total","panel_unique_types","matelines_total"]
minimize_cols = ["circ_cost","panel_count_total","panel_unique_types","matelines_total"]

df = pd.read_excel(excel_path, sheet_name=sheet_name)
df = df[cols].dropna().copy()
df = df.sort_values("score", ascending=False).reset_index(drop=True)

best_score_idx = 0
best_panel_idx = (df["panel_count_total"] + df["matelines_total"]).idxmin()

df_norm = pd.DataFrame(index=df.index)
for c in cols:
    v = df[c].values.astype(float)
    v_min, v_max = v.min(), v.max()
    v_norm = np.zeros_like(v) if abs(v_max - v_min) < 1e-9 else (v - v_min) / (v_max - v_min)
    if c in minimize_cols:
        v_norm = 1.0 - v_norm
    df_norm[c] = v_norm

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"]  = 42

x_positions = np.arange(len(cols))
fig, ax = plt.subplots(figsize=(11, 4.5), dpi=300, constrained_layout=True)

for idx, row in df_norm.iterrows():
    y = row.values
    if idx == best_score_idx:
        ax.plot(x_positions, y, linewidth=2.6, alpha=0.95, zorder=3)
    elif idx == best_panel_idx:
        ax.plot(x_positions, y, linewidth=2.2, alpha=0.90, zorder=2)
    else:
        ax.plot(x_positions, y, linewidth=0.8, alpha=0.25, zorder=1)

pretty = {
 "score":"Score",
 "area_public":"Public area",
 "area_private":"Private area",
 "circ_cost":"Circulation",
 "panel_count_total":"#Panels",
 "panel_unique_types":"#Panel types",
 "matelines_total":"#Matelines",
}

ax.set_xticks(x_positions)
ax.set_xticklabels([pretty.get(c,c) for c in cols], rotation=25, ha="right")
ax.set_ylim(0, 1)
ax.set_ylabel("Normalized metric (0–1)")
ax.set_title("Parallel Coordinates — GA Layout Metrics (normalized 0–1)")
ax.grid(True, axis="y", linestyle="--", linewidth=0.5, alpha=0.5)

legend_lines = [
    Line2D([0], [0], lw=2.6, label="Best score layout"),
    Line2D([0], [0], lw=2.2, label="Best panelization layout"),
    Line2D([0], [0], lw=0.8, alpha=0.6, label="Other layouts"),
]
ax.legend(handles=legend_lines, loc="upper right", fontsize=8)

save_figure(fig, "ga_parallel_coordinates", dpi=600)
plt.show()

***10.14: Benchmark Summary***

In [ ]:
excel_path = "ga_layout_metrics.xlsx"
sheet_name = "layouts_summary"
algorithm_label = "Genetic Algorithm (GA)"
runtime_seconds = float(run_info.get("runtime_sec", np.nan))

df = pd.read_excel(excel_path, sheet_name=sheet_name)

# ensure columns exist
if "compactness" not in df.columns:
    df["compactness"] = df["unused_fraction"] if "unused_fraction" in df.columns else np.nan
if "zoning_pen" not in df.columns:
    df["zoning_pen"] = 0.0

cols_needed = [
    "score","area_public","area_private","circ_cost","compactness","egress_pen","zoning_pen",
    "panel_count_total","panel_unique_types","matelines_total"
]
df = df[cols_needed].dropna(subset=["score"]).copy()
N = len(df)
print(f"{algorithm_label}: {N} layouts loaded from {excel_path}")

def pareto_front_df(df, max_cols, min_cols, eps=1e-9):
    vectors = []
    for idx, row in df.iterrows():
        v = {}
        for c in max_cols:
            v[c] = float(row[c])
        for c in min_cols:
            v[c] = -float(row[c])
        vectors.append((idx, v))

    front_idx = []
    for i, (idx_i, v_i) in enumerate(vectors):
        dominated = False
        for j, (idx_j, v_j) in enumerate(vectors):
            if i == j:
                continue
            ge = all(v_j[k] >= v_i[k] - eps for k in v_i)
            gt = any(v_j[k] > v_i[k] + eps for k in v_i)
            if ge and gt:
                dominated = True
                break
        if not dominated:
            front_idx.append(idx_i)
    return df.loc[front_idx].copy()

arch_max = ("area_public","area_private")
arch_min = ("circ_cost","compactness","egress_pen","zoning_pen")
panel_max = tuple()
panel_min = ("panel_count_total","panel_unique_types","matelines_total")

arch_front = pareto_front_df(df, arch_max, arch_min)
panel_front = pareto_front_df(df, panel_max, panel_min)

mean_score = df["score"].mean()
std_score  = df["score"].std(ddof=1)
cv_score   = std_score / mean_score if mean_score != 0 else np.nan

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"]  = 42

fig, axes = plt.subplots(2, 2, figsize=(11, 8), dpi=300, constrained_layout=True)

# (1) sorted score curve
ax = axes[0, 0]
scores_sorted = np.sort(df["score"].values)[::-1]
ax.plot(np.arange(1, len(scores_sorted) + 1), scores_sorted, lw=1.2)
ax.set_xlabel("Layout rank (best to worst)")
ax.set_ylabel("Score")
ax.set_title("Score curve (distribution / stability)")
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)

txt = f"N={N}\nμ={mean_score:.2f}\nσ={std_score:.2f}\nCV={cv_score:.2f}"
if np.isfinite(runtime_seconds):
    txt += f"\nTime={runtime_seconds:.1f}s"
ax.text(
    0.98, 0.98, txt, transform=ax.transAxes,
    ha="right", va="top",
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.85),
    fontsize=8
)

# (2) histogram
ax = axes[0, 1]
ax.hist(df["score"].values, bins=10, edgecolor="black", alpha=0.7)
ax.set_xlabel("Score")
ax.set_ylabel("Count")
ax.set_title("Score histogram")
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

# (3) architectural scatter
ax = axes[1, 0]
ax.scatter(df["area_public"], df["circ_cost"], s=22, alpha=0.55, label="All layouts",
           edgecolors="black", linewidths=0.25)
ax.scatter(arch_front["area_public"], arch_front["circ_cost"], s=45, alpha=0.9,
           label="Architectural Pareto", edgecolors="black", linewidths=0.35)
ax.set_xlabel("Public area (m²)")
ax.set_ylabel("Circulation cost (m)")
ax.set_title("Architectural trade-off")
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
ax.legend(fontsize=8, framealpha=0.9)

# (4) panelization scatter
ax = axes[1, 1]
ax.scatter(df["panel_count_total"], df["matelines_total"], s=22, alpha=0.55, label="All layouts",
           edgecolors="black", linewidths=0.25)
ax.scatter(panel_front["panel_count_total"], panel_front["matelines_total"], s=45, alpha=0.9,
           label="Panelization Pareto", edgecolors="black", linewidths=0.35)
ax.set_xlabel("Total panels")
ax.set_ylabel("Matelines")
ax.set_title("Panelization trade-off")
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
ax.legend(fontsize=8, framealpha=0.9)

fig.suptitle(f"{algorithm_label} — Benchmark Summary", fontsize=14)
save_figure(fig, "ga_benchmark", dpi=600)
plt.show()

***10.15: Visualize and Save GA Solutions***

In [65]:
import os, math, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from shapely.geometry import Polygon, Point
from shapely.ops import unary_union

# -----------------------------
# Helpers: geometry from cells
# -----------------------------
def _cell_poly(G_cells, node):
    cx, cy = G_cells.nodes[node]['coord']
    w, h   = G_cells.nodes[node]['size']
    return Polygon([
        (cx - w/2, cy - h/2),
        (cx + w/2, cy - h/2),
        (cx + w/2, cy + h/2),
        (cx - w/2, cy + h/2),
    ])

def get_room_polygon(G_cells, nodes):
    if not nodes:
        return Polygon()
    return unary_union([_cell_poly(G_cells, n) for n in nodes])

def base_of(uid: str) -> str:
    return uid.split("#", 1)[0].strip()

# -----------------------------
# Fallback panelization (only if Step-9 panelize_layout is absent/fails)
# -----------------------------
def _collect_wall_segments(G_cells, layout_uid_dict):
    node2room = {}
    for uid, nodes in layout_uid_dict.items():
        for n in nodes:
            node2room[n] = uid

    center2node = {tuple(G_cells.nodes[n]['coord']): n for n in G_cells}
    w, h = G_cells.nodes[next(iter(G_cells.nodes))]['size']
    OFF = {"L": (-w, 0), "R": (w, 0), "B": (0, -h), "T": (0, h)}

    def ek(p0, p1):
        p0 = (float(p0[0]), float(p0[1]))
        p1 = (float(p1[0]), float(p1[1]))
        return tuple(sorted((p0, p1)))

    segs = set()
    for n, uid in node2room.items():
        cx, cy = G_cells.nodes[n]['coord']
        x0, y0, x1, y1 = cx - w/2, cy - h/2, cx + w/2, cy + h/2

        faces = {
            "L": ((x0, y0), (x0, y1)),
            "R": ((x1, y0), (x1, y1)),
            "B": ((x0, y0), (x1, y0)),
            "T": ((x0, y1), (x1, y1)),
        }

        for k, (pA, pB) in faces.items():
            dx, dy = OFF[k]
            nbr_center = (round(cx + dx, 6), round(cy + dy, 6))
            nbr = center2node.get(nbr_center)
            if (nbr is None) or (node2room.get(nbr) != uid):
                segs.add(ek(pA, pB))

    return [tuple(s) for s in segs]

def _merge_collinear_segments(segments, tol=1e-6):
    horiz, vert = {}, {}
    for (p0, p1) in segments:
        x0, y0 = p0; x1, y1 = p1
        if abs(y0 - y1) < tol:
            y = round(y0, 6)
            a, b = sorted([x0, x1])
            horiz.setdefault(y, []).append((a, b))
        elif abs(x0 - x1) < tol:
            x = round(x0, 6)
            a, b = sorted([y0, y1])
            vert.setdefault(x, []).append((a, b))

    def merge(intervals):
        if not intervals:
            return []
        intervals.sort()
        out = [list(intervals[0])]
        for a, b in intervals[1:]:
            if a <= out[-1][1] + 1e-9:
                out[-1][1] = max(out[-1][1], b)
            else:
                out.append([a, b])
        return [tuple(x) for x in out]

    runs = []
    for y, xs in horiz.items():
        for a, b in merge(xs):
            runs.append(((a, y), (b, y)))
    for x, ys in vert.items():
        for a, b in merge(ys):
            runs.append(((x, a), (x, b)))
    return runs

def fallback_panelize_layout(layout, G_cells):
    runs = _merge_collinear_segments(_collect_wall_segments(G_cells, layout))
    # joints: endpoints
    endpoints = []
    seen = set()
    for p0, p1 in runs:
        for p in (p0, p1):
            key = (round(p[0], 6), round(p[1], 6))
            if key not in seen:
                seen.add(key)
                endpoints.append({"x": p[0], "y": p[1]})
    # estimate length in cells
    w, h = G_cells.nodes[next(iter(G_cells.nodes))]['size']
    module = min(float(w), float(h))
    panels = []
    for i, (p0, p1) in enumerate(runs, start=1):
        Lm = math.hypot(p1[0] - p0[0], p1[1] - p0[1])
        length_cells = int(round(Lm / module)) if module > 0 else 0
        panels.append({
            "id": f"P{i}",
            "p0": p0,
            "p1": p1,
            "length_cells": length_cells
        })
    return {
        "summary": {"total_panels": len(panels), "total_matelines": len(endpoints)},
        "panels": panels,
        "matelines": endpoints
    }

def safe_panelize_layout(layout, G_cells):
    # Prefer your Step 9 function if available
    try:
        pz = panelize_layout(layout, G_cells)
        # normalize minimal expected keys
        pz.setdefault("summary", {})
        pz.setdefault("panels", [])
        pz.setdefault("matelines", [])
        return pz
    except Exception:
        return fallback_panelize_layout(layout, G_cells)

# -----------------------------
# Plot: Step-9-like view with room labels + panels + joints
# -----------------------------
def plot_layout_with_panels_and_rooms(
    layout: Dict[str, List[str]],
    G_cells,
    building_polygon,
    plot_polygon,
    title="",
    annotate_panels=True,
    annotate_rooms=True,
):
    pz = safe_panelize_layout(layout, G_cells)
    sm = pz.get("summary", {})
    panels = pz.get("panels", [])
    matelines = pz.get("matelines", [])

    # Color map for rooms
    room_uids = list(layout.keys())
    cmap_rooms = plt.cm.get_cmap("tab20", max(1, len(room_uids)))

    # Color map for panel lengths (len=1,2,3,...)
    # We'll map unique length_cells to a small set of colors using tab10
    lengths = [int(p.get("length_cells", 0)) for p in panels]
    uniq_lengths = sorted(set(lengths))
    cmap_pan = plt.cm.get_cmap("tab10", max(1, len(uniq_lengths)))
    len2color = {L: cmap_pan(i) for i, L in enumerate(uniq_lengths)}

    fig, ax = plt.subplots(figsize=(6.8, 4.6), dpi=300, constrained_layout=True)

    # plot + building background
    ax.add_patch(patches.Polygon(np.array(plot_polygon.exterior.coords),
                                 fill=False, ec="gray", ls="--", lw=1.2))
    ax.add_patch(patches.Polygon(np.array(building_polygon.exterior.coords),
                                 fc="lightgray", ec="none", alpha=0.35))

    # draw rooms as cell rectangles
    for i, uid in enumerate(room_uids):
        col = cmap_rooms(i)
        for n in layout[uid]:
            cx, cy = G_cells.nodes[n]["coord"]
            w, h = G_cells.nodes[n]["size"]
            ax.add_patch(
                patches.Rectangle((cx - w/2, cy - h/2), w, h,
                                  fc=col, ec="black", lw=0.35, alpha=0.55)
            )

        if annotate_rooms:
            poly = get_room_polygon(G_cells, layout[uid])
            if not poly.is_empty:
                c = poly.centroid
                ax.text(c.x, c.y, uid, fontsize=7, ha="center", va="center",
                        bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.6))

    # draw panels
    for idx, p in enumerate(panels, start=1):
        p0 = p.get("p0", None); p1 = p.get("p1", None)
        if p0 is None or p1 is None:
            # try alternative keys some panelizers use
            p0 = p.get("start", None)
            p1 = p.get("end", None)
        if p0 is None or p1 is None:
            continue

        Lc = int(p.get("length_cells", 0))
        col = len2color.get(Lc, (0, 0, 0, 1))
        ax.plot([p0[0], p1[0]], [p0[1], p1[1]], lw=2.2, color=col, solid_capstyle="round", zorder=5)

        if annotate_panels:
            mx, my = (p0[0]+p1[0])/2, (p0[1]+p1[1])/2
            pid = p.get("id", f"P{idx}")
            ax.text(mx, my, str(pid), fontsize=6.5, ha="center", va="center",
                    bbox=dict(boxstyle="round,pad=0.1", fc="white", ec="none", alpha=0.6),
                    zorder=6)

    # draw joints/matelines (magenta X)
    for m in matelines:
        if isinstance(m, dict):
            x = m.get("x", None); y = m.get("y", None)
            if x is None and "pt" in m:
                x, y = m["pt"]
        else:
            # tuple/list
            x, y = m[0], m[1]
        if x is None or y is None:
            continue
        ax.scatter([x], [y], marker="x", s=55, linewidths=2.0, color="magenta", zorder=7)

    # title with summary
    total_panels = sm.get("total_panels", len(panels))
    total_mates  = sm.get("total_matelines", len(matelines))
    ax.set_title(f"{title} | panels={total_panels} | matelines={total_mates}", fontsize=10)

    xmin, ymin, xmax, ymax = plot_polygon.bounds
    pad = 0.6
    ax.set_xlim(xmin - pad, xmax + pad)
    ax.set_ylim(ymin - pad, ymax + pad)
    ax.set_aspect("equal")
    ax.grid(True, lw=0.3, alpha=0.35)
    ax.tick_params(labelsize=8)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")

    # legend for panel lengths (len=... cells)
    if uniq_lengths:
        handles = []
        for L in uniq_lengths:
            handles.append(plt.Line2D([0], [0], color=len2color[L], lw=2.2, label=f"len={L} cells"))
        ax.legend(handles=handles, loc="upper right", fontsize=7, framealpha=0.9)

    return fig, ax

# -----------------------------
# Save all GA layouts
# -----------------------------
def save_all_ga_layout_images(
    ga_sorted=None,
    excel_path="ga_layout_metrics.xlsx",
    out_dir="ga_optimized_layout_images",
    top_n=None,
    annotate_panels=True,
    annotate_rooms=True
):
    os.makedirs(out_dir, exist_ok=True)

    # If ga_sorted is not passed, try to reconstruct layouts from memory is not possible via Excel alone.
    # So: prefer ga_sorted. Excel is mainly for metrics, not for layout geometry.
    if ga_sorted is None:
        raise ValueError(
            "Please pass ga_sorted (the list of individuals with 'layout' dict). "
            "Excel file does not contain cell allocations, so layouts cannot be reconstructed from it."
        )

    n_total = len(ga_sorted)
    if top_n is None:
        top_n = n_total
    top_n = min(top_n, n_total)

    print(f"Saving {top_n}/{n_total} GA optimized layouts to: {out_dir}")

    for i in range(top_n):
        ind = ga_sorted[i]
        layout = ind["layout"]
        logic = ind.get("logic", None)
        score = ind.get("score", None)

        title = f"GA#{i:03d} | score={score:.3f}" if score is not None else f"GA#{i:03d}"
        if logic is not None:
            title += f" | L={logic}"

        fig, ax = plot_layout_with_panels_and_rooms(
            layout=layout,
            G_cells=G_cells,
            building_polygon=selected_building,
            plot_polygon=spatial_data["polygon"],
            title=title,
            annotate_panels=annotate_panels,
            annotate_rooms=annotate_rooms,
        )

        fname = os.path.join(out_dir, f"GA_layout_{i:03d}_L{logic}_score_{score:.3f}.png")
        # Use your helper if available, else fallback
        try:
            save_figure(fig, os.path.splitext(os.path.basename(fname))[0], dpi=600)
            # If your save_figure saves elsewhere, also save locally:
            fig.savefig(fname, dpi=600)
        except Exception:
            fig.savefig(fname, dpi=600)

        plt.close(fig)

    print("✅ Done saving GA layout images.")

In [ ]:
# Save ALL optimized solutions (or set top_n=50 for first 50)
save_all_ga_layout_images(
    ga_sorted=ga_sorted,            # <-- from your Step 10 GA run
    out_dir="ga_optimized_layout_images",
    top_n=None,                     # None = all
    annotate_panels=True,
    annotate_rooms=True
)